# Multi-Task Activity Classification Pipeline for PSMA Inhibitors (F7)

Predicts whether a compound is a potent PSMA (glutamate carboxypeptidase II / FOLH1)
inhibitor on **two endpoints jointly** — pIC50 and pKi, each as a binary active /
not-active call at 10 nM — and ranks a screening library by P(active). No pretraining:
the network trains directly on the PSMA data from random initialisation.

## What changed from F6, and why

**1. Multi-task is back, because single-task was throwing away a third of the data.**
F6 modelled pIC50 alone, on the grounds that 139 of the 169 dual-measured compounds have
pKi arithmetically derived from pIC50. That reasoning was correct about the *overlap* and
silently wrong about everything else: of 681 compounds, 283 carry only pIC50, 169 carry
both, and **229 carry only pKi**. A pIC50-only model cannot see those 229 at all. Those
compounds are also the ones least likely to be contaminated — with no IC50 in the record,
there is nothing for a Ki to have been derived from.

**2. The derived values are deleted rather than thresholded around.** The pKi column mixes
two incompatible scales: for the derived block pKi = pIC50 + 2.145 (IC50/Ki = 140.0
exactly), so a 10 nM-equivalent active sits at pKi ≥ 10.15, while for a genuinely measured
Ki it sits at pKi ≥ 8.0. No single threshold is right for both, which is what produced the
50–67% endpoint disagreement in Part A.2b. Since a derived value is just pIC50 ÷ 140 and
carries no information, `drop_derived_endpoint_values` removes those pKi labels (keeping
each compound's pIC50), after which one pharmacological cut is correct everywhere.

The result is **712 usable compounds against F6's 483** — a 47% increase, and the largest
single improvement available to this pipeline. More and better-spread data has outrun
every modelling change tried across F3–F6.

**3. No pretraining.** Measured twice now, in opposite directions and never materially:
F4's regression found +0.062 R² on random splits (10/10 repeats, p = 0.002) and nothing on
either grouped scheme; F6's classification found −0.0066 AUC on random splits (1/10
repeats, p = 0.020) and nothing under leave-one-cluster-out. Effect sizes under 0.01 AUC,
sign depending on the objective. `use_pretraining = False` is now the default, so the
trunk starts from random init and the frozen/unfrozen schedule is skipped. The code path
is retained rather than deleted, and `run_pretraining_ablation` re-measures it on demand —
the claim stays reproducible instead of becoming an assertion.

**4. Probabilities are recalibrated.** F6's were ~2× over-stated: mean predicted 0.50
against an observed rate of 0.265, ECE 0.24–0.31, caused by the balanced class weighting
pushing outputs toward 0.5. Part E.2e now fits an isotonic (or Platt) calibrator
**cross-fitted across the leave-one-cluster-out folds** — each fold calibrated on the
other folds' out-of-fold predictions, never its own — and reports ECE and Brier before and
after. The fitted calibrator is then applied to the library scores, so P(active) means
what it says.

**5. Folds are large enough to score.** F6 reported eight folds of which four were
uninterpretable: cluster 6 had zero actives (every metric NaN), cluster 4 had one active
in 18 compounds (AUC 0.059 — one compound landing low in a ranking, not a measurement),
cluster 1 had 14 compounds at 71% actives (AUC 1.000, nearly free at that base rate). A
fold is now reported individually only if it clears both `loco_min_cluster_size` and
`loco_min_fold_actives`; the rest are distributed into `loco_n_remainder_folds`
size-balanced "small families" folds. Every compound still gets an out-of-fold prediction,
so pooled coverage stays complete — only the per-fold table changes, and it becomes
readable.

**6. Library and training duplicates are merged on InChIKey.** F6's shortlist of 25 rows
contained `sdf:199278`, `sdf:818`, `sdf:67406` and `sdf:94329` twice each and `sdf:92796`
three times — LigPrep protonation and tautomer variants of single compounds. Deduplication
now happens on the InChIKey of the **RDKit-standardised** structure, in both the screening
library and the training set, which also closes the long-standing issue of training
compounds being deduplicated on raw SMILES before standardisation.

## Carried forward from F6

The **Tanimoto veto** on the applicability domain (`ad_tanimoto_veto`), which revealed that
only 36 of 289,554 library compounds are within structural reach of the training set —
F6's headline finding, and still the binding constraint on this whole exercise. The
**equal-weight probability blend**, since fitted weights lost 6/6 in F5. **Enrichment
factor and precision@N**, because "of my top 50, how many are real" is the only question a
screen exists to answer. And **leave-one-cluster-out** as the evaluation of record, with a
stratified random split alongside purely to keep the leakage gap visible.

A two-model blend (LightGBM + KPLS-DA) is added to the committee comparison, because in F6
KPLS-DA alone beat the three-way average on the honest evaluation (AUC 0.706 vs 0.690,
precision@50 0.58 vs 0.52) and it is worth knowing whether the network is diluting it.

## Pre-registered thresholds

- **active: pIC50 ≥ 8.0** (IC50 ≤ 10 nM)
- **active: pKi ≥ 8.0** (Ki ≤ 10 nM)

Both are the same pharmacological potency, which is only coherent because the derived
values have been removed. Set before any model was fitted.

## What this still does not fix

The screening library remains the wrong file — fragments and solvents, and with 100% of it
failing the structural test, no modelling change in this notebook can produce a credible
shortlist from it. Nothing has been tested in a lab. And 712 compounds from a handful of
patent families is still a small, narrow dataset; F7 measures that more clearly than F6
did rather than solving it.

## 0. Setup

Install dependencies. Safe to re-run — pip skips anything already satisfied.


In [ ]:
%pip install -q duckdb rdkit pandas numpy scikit-learn matplotlib seaborn requests tqdm scipy joblib \
    lightgbm optuna scikit-optimize torch chembl_webresource_client


In [ ]:
import os
import io
import re
import json
import zipfile
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Sequence, List, Dict, Tuple

import duckdb
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs

from sklearn.model_selection import train_test_split, KFold, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import Nystroem
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import make_scorer
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib

import torch
import torch.nn as nn

import optuna
from skopt import BayesSearchCV
from skopt.space import Integer, Real

from lightgbm import LGBMRegressor

import matplotlib.pyplot as plt
import seaborn as sns

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="notebook")

np.random.seed(0)
torch.manual_seed(0)

# ---- GPU support -----------------------------------------------------------
# Every NN training/inference function below (`pretrain_trunk`, `finetune_multitask`,
# `predict_with_mc_dropout`) accepts this `DEVICE` and moves its model/tensors onto it,
# so the entire pretrain -> fine-tune -> MC-dropout pipeline runs on GPU automatically
# when one is available (CUDA, or Apple Silicon MPS), and falls back to CPU otherwise.
# On Colab: Runtime -> Change runtime type -> select a GPU, then re-run from the top.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU detected: {torch.cuda.get_device_name(0)} -> using CUDA.")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Apple Silicon GPU detected -> using MPS.")
else:
    DEVICE = torch.device("cpu")
    print("No GPU detected -> using CPU. (Colab: Runtime > Change runtime type > GPU.)")


def rmse(y_true, y_pred) -> float:
    '''Root-mean-squared-error, computed manually so this works across sklearn versions
    (newer sklearn removed mean_squared_error's `squared=False` argument).'''
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


print("Environment ready.")


## 1. Configuration

Everything that changes between projects lives here: ChEMBL pretraining scope, the
BindingDB target/endpoints, screening library path, AD thresholds, model
architectures, and every hyperparameter-search setting. Nothing below this cell
needs to change for a routine re-run.


In [ ]:
@dataclass
class Config:
    # ---- ChEMBL pretraining set (domain-targeted by default) -------------------
    # A bioactivity set used only to *pretrain* the shared NN trunk (transfer
    # learning) before it ever sees PSMA data. Pulled live via
    # chembl_webresource_client; cached to `chembl_pretrain_cache` so re-runs are free.
    #
    # "domain_targeted" (default): restrict pretraining to targets related to PSMA —
    # PSMA (GCPII/FOLH1) is a zinc-dependent, co-catalytic-dinuclear-zinc metallopeptidase
    # (MEROPS clan MH, family M28). Other zinc metallopeptidases/metalloproteases are
    # the natural "related" pool: their inhibitors are built around the same handful of
    # zinc-binding chemotypes PSMA inhibitors use (phosphonates, thiols, hydroxamates,
    # ureas), so pretraining here teaches the trunk that specific structure-activity
    # vocabulary rather than generic, unrelated bioactivity patterns.
    # "broad": the original target-agnostic sample across arbitrary human targets.
    chembl_pretrain_mode: str = "domain_targeted"     # "domain_targeted" | "broad"
    chembl_pretrain_target_keywords: Sequence[str] = field(default_factory=lambda: [
        "glutamate carboxypeptidase",       # PSMA's own family (paralogs, e.g. NAALADase L)
        "aminopeptidase",                   # other M-clan zinc peptidases, same catalytic chemistry
        "leucine aminopeptidase",
        "carboxypeptidase",
        "angiotensin-converting enzyme",    # zinc metallopeptidase; classic phosphonate/thiol ZBGs
        "neprilysin",                       # zinc metallopeptidase (NEP), thiol/phosphonate inhibitors
        "endothelin-converting enzyme",     # related M13 zinc metallopeptidase
        "thermolysin",                      # archetypal zinc metallopeptidase used to design ZBGs
        "matrix metalloproteinase",         # zinc metalloproteinases, hydroxamate ZBGs
        "insulin-degrading enzyme",         # zinc metallopeptidase
        "meprin",                           # zinc metalloproteinase
        "carbonic anhydrase",               # different fold, but a classic zinc metalloenzyme drug
                                             # target with a rich, structurally-diverse zinc-binding
                                             # inhibitor literature (sulfonamides, etc.)
    ])
    chembl_pretrain_target_type: str = "SINGLE PROTEIN"
    chembl_pretrain_min_targets_or_fallback_to_broad: int = 5   # if fewer targets match, fall back to "broad"
    chembl_pretrain_cache: str = "./ad_pipeline_data/chembl_pretrain_domain.csv"
    chembl_pretrain_standard_types: Sequence[str] = field(
        default_factory=lambda: ["IC50", "Ki", "Kd", "EC50"]
    )
    chembl_pretrain_organism: str = "Homo sapiens"
    chembl_pretrain_max_targets: int = 150      # cap on distinct targets sampled ("broad" mode only)
    chembl_pretrain_per_target_cap: int = 300   # max activities pulled per target
    chembl_pretrain_max_compounds: int = 10000   # hard cap on the final pretraining set
    chembl_pretrain_endpoint_name: str = "pX_generic"   # generic potency label used only for pretraining
    force_rebuild_chembl_cache: bool = False

    # ---- BindingDB acquisition (PSMA fine-tuning set) --------------------------
    bindingdb_tsv_zip_name: str = "BindingDB_All_202609_tsv.zip"
    bindingdb_download_url: str = (
        "https://www.bindingdb.org/rwd/bind/downloads/BindingDB_All_202609_tsv.zip"
    )
    auto_resolve_bindingdb_url: bool = True
    data_dir: str = "./ad_pipeline_data"
    force_redownload: bool = False

    # ---- Target / multi-task endpoint selection (BindingDB SQL filter) --------
    target_uniprot_id: Optional[str] = "Q04609"          # PSMA / FOLH1 / GCPII
    target_name_like: Optional[str] = "Glutamate carboxypeptidase 2"
    organism_like: Optional[str] = "Homo sapiens"

    # Multi-task: model BOTH endpoints jointly instead of picking one.
    affinity_columns: Sequence[str] = field(default_factory=lambda: ["IC50 (nM)", "Ki (nM)"])
    endpoint_names: Sequence[str] = field(default_factory=lambda: ["pIC50", "pKi"])

    max_affinity_nm: float = 1e6
    min_affinity_nm: float = 1e-3

    # ---- Curated screening library ----------------------------------------------
    screening_library_path: str = "/Enamine_screen_library.sdf"
    screening_smiles_col: str = "SMILES"
    screening_id_col: Optional[str] = "compound_id"

    # ---- Featurization ------------------------------------------------------------
    morgan_radius: int = 2
    morgan_n_bits: int = 2048
    feature_set: str = "fp_plus_descriptors"   # "fp_plus_descriptors" | "descriptors_only" | "fp_only"

    # ---- Applicability domain thresholds (calibrated on the PSMA fine-tuning set) --
    ad_percentile_threshold: float = 95.0
    unfamiliarity_k: int = 5
    latent_pca_components: int = 20

    # ---- Neural-network transfer-learning model ------------------------------------
    nn_hidden_dims: Sequence[int] = field(default_factory=lambda: (256, 128))
    nn_dropout: float = 0.2
    nn_lr: float = 1e-3
    nn_finetune_lr: float = 5e-4
    nn_weight_decay: float = 1e-5
    nn_pretrain_epochs: int = 40
    nn_finetune_total_epochs: int = 80
    nn_freeze_epochs: int = 15          # epochs of head-only training before unfreezing the trunk
    nn_batch_size: int = 32
    nn_pretrain_batch_size: int = 128
    nn_mc_dropout_passes: int = 30      # forward passes for MC-dropout uncertainty

    # ---- LightGBM baseline (trained directly on BindingDB, one model per task) -----
    n_bootstrap: int = 30               # bootstrap ensemble size for GBM uncertainty

    # ---- Hyperparameter search ------------------------------------------------------
    run_gridsearch: bool = True
    run_bayes_search: bool = True       # scikit-optimize BayesSearchCV
    run_optuna: bool = True             # Optuna TPE search for the NN
    bayes_search_n_iter: int = 20
    optuna_n_trials: int = 25
    optuna_cv_folds: int = 3            # cheap inner CV used inside the Optuna objective

    test_size: float = 0.2
    cv_folds: int = 10
    random_state: int = 42
    n_eval_repeats: int = 10          # repeated random splits for robust held-out statistics
    n_calibration_bins: int = 5       # uncertainty-calibration reliability diagram bins

    # ---- Output ----------------------------------------------------------------------
    output_dir: str = "./ad_pipeline_outputs/f7"


    # ================================================================================
    # F6: single-task activity classification
    # ================================================================================
    # The data pull below still fetches BOTH endpoints, because Part A.2b's diagnostic
    # needs them to demonstrate *why* pKi is dropped. Everything from Part A.2c onward
    # models `primary_endpoint` alone.
    task_mode: str = "classification"        # "classification" | "regression"

    # ---- F7: multi-task over both endpoints -------------------------------------
    # F6 modelled pIC50 alone and so never saw the 229 compounds that carry only a Ki
    # measurement - a third of the data. Both endpoints are modelled jointly again, with
    # a masked loss so a compound needs only one of the two labels to contribute.
    multitask: bool = True
    modelled_endpoints: Sequence[str] = field(default_factory=lambda: ["pIC50", "pKi"])
    primary_endpoint: str = "pIC50"          # used for ranking and single-number summaries
    primary_affinity_column: str = "IC50 (nM)"

    # Per-endpoint active cut, on the pAffinity scale. Both are 10 nM, i.e. the SAME
    # pharmacological potency - which is only coherent once the derived Ki values are
    # removed (see drop_derived_endpoint_values).
    active_thresholds: Dict[str, float] = field(
        default_factory=lambda: {"pIC50": 8.0, "pKi": 8.0}
    )

    # ---- Derived-value removal ---------------------------------------------------
    # Part A.2b found 139 of 169 dual-measured compounds sitting in fixed-offset blocks:
    # 111 at IC50/Ki = 140.0 (US20250213705) and 28 at 5.01 (US10736974). Those Ki values
    # were computed from the IC50 values by Cheng-Prusoff, not measured. They carry no
    # information AND they break any single pKi threshold, so the pKi label is dropped for
    # those compounds while their pIC50 is kept.
    drop_derived_endpoint_values: bool = True
    derived_min_block_size: int = 5          # compounds sharing an offset before it counts
    derived_offset_tol: float = 0.02         # log units

    # ---- Transfer learning: OFF ---------------------------------------------------
    # Measured twice. F4 regression: +0.062 R2 on random splits (10/10, p=0.002), nothing
    # on grouped. F6 classification: -0.0066 AUC on random splits (1/10, p=0.020), nothing
    # on grouped. Effects under 0.01 AUC with the sign flipping by objective. The code
    # path is kept so the claim stays reproducible, not deleted.
    use_pretraining: bool = False
    run_pretraining_ablation: bool = False   # True re-measures it (costs one pretrain pass)

    # ---- Probability recalibration ------------------------------------------------
    # F6 predicted ~0.50 where the observed rate was 0.265 (ECE 0.24-0.31) because the
    # balanced class weighting pushes outputs toward 0.5. Cross-fitted across the LOCO
    # folds: each fold is calibrated on the OTHER folds' out-of-fold predictions.
    # Platt (sigmoid) by default, not isotonic. The miscalibration being corrected is a
    # systematic ~2x inflation from the balanced class weighting, which is close to a pure
    # logit shift - exactly what a 2-parameter sigmoid models. Isotonic is more flexible
    # but fits a step function, and with only a handful of LOCO folds it overfits and
    # collapses distinct scores into ties, which destroys ranking resolution (a smoke test
    # on noise features moved AUC by 0.10 that way). Switch to "isotonic" if the
    # before/after table shows a miscalibration shape a sigmoid cannot follow.
    recalibration: str = "sigmoid"           # "sigmoid" (Platt) | "isotonic" | "none"

    # ---- Leave-one-cluster-out folds ----------------------------------------------
    # F6 reported 8 folds of which 4 could not be scored (0 or 1 actives, or a 71% base
    # rate on 14 compounds). A cluster now needs size AND actives to get its own row.
    loco_min_cluster_size: int = 25
    loco_min_fold_actives: int = 5
    loco_n_remainder_folds: int = 3          # size-balanced folds for everything smaller

    # ---- Deduplication -------------------------------------------------------------
    # On the InChIKey of the RDKit-STANDARDISED structure, so LigPrep protonation and
    # tautomer variants collapse. Applies to the library and the training set; the latter
    # closes the old issue of deduplicating on raw SMILES before standardisation.
    dedupe_on_inchikey: bool = True

    # Pre-registered class boundaries on the pAffinity scale. Anchored on pharmacology
    # (10 nM / 1 uM), NOT on quantiles of this particular dataset - a quantile cut makes
    # "active" mean "top third of this snapshot", which is not transferable and shifts
    # when data is added.
    n_activity_classes: int = 2              # 2 = active vs not-active; 3 = + moderate
    active_threshold: float = 8.0            # pIC50 >= 8.0  <=>  IC50 <= 10 nM
    inactive_threshold: float = 6.0          # pIC50 <  6.0  <=>  IC50 >  1 uM (3-class only)
    keep_censored_as_labels: bool = True     # recover '>' rows as inactives, '<' as actives
    class_weight_balanced: bool = True       # class-weighted loss / fitting

    # Exploratory: down-weight over-represented sources (one patent is 26% of the
    # pIC50 labels). Inverse-sqrt-frequency per source tag. Default OFF - this is a
    # modelling choice with no obviously right answer, exposed so its effect is
    # measurable rather than assumed.
    downweight_dominant_source: bool = False

    # ---- Applicability domain ---------------------------------------------------
    # F5's consensus rule let compounds pass while failing the Tanimoto test, which for a
    # congeneric target is the flag with teeth: all 15 F5 top hits were tanimoto_OOD and
    # all 15 passed. With the veto on, failing Tanimoto alone puts a compound out of
    # domain regardless of the other three metrics.
    ad_tanimoto_veto: bool = True
    ad_consensus_min_flags: int = 3

    # ---- Leave-one-cluster-out evaluation ---------------------------------------
    butina_cutoff: float = 0.4               # Tanimoto distance cutoff for clustering
    loco_max_clusters: int = 12              # largest N clusters get their own fold
    loco_min_cluster_size: int = 12          # smaller clusters are pooled into one fold
    loco_random_reference_repeats: int = 10  # random-split repeats, for comparison only

    # ---- Screening metrics -------------------------------------------------------
    enrichment_fractions: Sequence[float] = field(
        default_factory=lambda: [0.01, 0.05, 0.10]
    )
    precision_at_n: Sequence[int] = field(default_factory=lambda: [50, 100, 200])

    def __post_init__(self):
        Path(self.data_dir).mkdir(parents=True, exist_ok=True)
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


CONFIG = Config()
CONFIG

## 2. Part A.1 — ChEMBL pretraining set: PSMA-related targets only

Rather than pretraining on an arbitrary cross-section of ChEMBL, this pulls
bioactivities **only from targets related to PSMA** — controlled by
`CONFIG.chembl_pretrain_target_keywords`. PSMA (GCPII/FOLH1) is a zinc-dependent,
co-catalytic-dinuclear-zinc **metallopeptidase** (MEROPS family M28), and the great
majority of known PSMA inhibitors are built around a handful of zinc-binding groups
(phosphonates, thiols, hydroxamates, ureas) shared with other zinc metallopeptidase
inhibitors. So the "related" pool queried here is: PSMA's own paralog family
(other glutamate carboxypeptidases), other aminopeptidases/carboxypeptidases,
and the classic zinc-metalloprotease drug targets whose inhibitor chemistry PSMA
inhibitors borrow from directly — ACE, neprilysin, ECE-1, thermolysin, MMPs,
insulin-degrading enzyme, meprin — plus carbonic anhydrase, included for its
structurally diverse zinc-binding inhibitor literature even though its fold differs.

This should transfer better than a target-agnostic broad set because the trunk
learns the specific structure-activity vocabulary of zinc-binding-group chemistry,
not generic bioactivity patterns from unrelated target classes. Cached to CSV so
repeated runs don't re-hit the API; if ChEMBL is unreachable, or too few related
targets are found, it falls back first to the old **broad** (target-agnostic) mode
and then to a small synthetic set, so the rest of the notebook always still runs.


In [ ]:
def _synthetic_chembl_pretrain_set(cfg: Config, n: int = 500) -> pd.DataFrame:
    '''Small, clearly-labeled synthetic stand-in used only if ChEMBL can't be reached.'''
    print("!! Using SYNTHETIC ChEMBL pretraining data (network/API unavailable) !!")
    demo_smiles = [
        "CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
        "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
        "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
        "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12",
        "Cc1ccc(cc1)S(=O)(=O)N", "O=C(Nc1ccccc1)c1ccccc1O",
        "CCN(CC)CCOC(=O)c1ccc(N)cc1", "COc1ccc2[nH]c(nc2c1)S(=O)Cc1ncc(C)c(OC)c1C",
    ]
    rng = np.random.RandomState(cfg.random_state)
    rows = []
    for i in range(n):
        base = demo_smiles[i % len(demo_smiles)]
        rows.append({
            "smiles": base,
            "target_chembl_id": f"SYNTH_TARGET_{i % 40}",
            cfg.chembl_pretrain_endpoint_name: float(np.clip(rng.normal(6.5, 1.5), 3.0, 10.0)),
        })
    return pd.DataFrame(rows)


def find_related_chembl_targets(cfg: Config) -> pd.DataFrame:
    '''Searches ChEMBL's target registry for every target whose preferred name matches
    one of `chembl_pretrain_target_keywords`, returning a deduplicated
    (target_chembl_id, pref_name, organism, matched_keyword) table.'''
    from chembl_webresource_client.new_client import new_client

    target_client = new_client.target
    rows = []
    for kw in cfg.chembl_pretrain_target_keywords:
        try:
            hits = target_client.filter(pref_name__icontains=kw, target_type=cfg.chembl_pretrain_target_type) \
                                 .only(["target_chembl_id", "pref_name", "organism"])
            for h in hits:
                rows.append({"target_chembl_id": h.get("target_chembl_id"), "pref_name": h.get("pref_name"),
                             "organism": h.get("organism"), "matched_keyword": kw})
        except Exception as e:
            print(f"  target search failed for keyword '{kw}': {e}")
    if not rows:
        return pd.DataFrame(columns=["target_chembl_id", "pref_name", "organism", "matched_keyword"])
    df = pd.DataFrame(rows).dropna(subset=["target_chembl_id"])
    df = df.drop_duplicates(subset=["target_chembl_id"]).reset_index(drop=True)
    return df


def fetch_activities_for_targets(cfg: Config, target_ids: Sequence[str]) -> pd.DataFrame:
    '''Pulls bioactivities restricted to a specific list of target_chembl_ids, in
    batches (long __in lists are chunked to keep each request well-formed).'''
    from chembl_webresource_client.new_client import new_client

    activity_client = new_client.activity
    rows = []
    per_target_counts: Dict[str, int] = {}
    batch_size = 40
    target_ids = list(target_ids)

    for i in range(0, len(target_ids), batch_size):
        batch = target_ids[i:i + batch_size]
        query = activity_client.filter(
            target_chembl_id__in=batch,
            standard_type__in=list(cfg.chembl_pretrain_standard_types),
            pchembl_value__isnull=False,
        ).only(["canonical_smiles", "target_chembl_id", "pchembl_value"])

        for rec in tqdm(query, desc=f"ChEMBL activities (targets {i+1}-{i+len(batch)}/{len(target_ids)})"):
            smi = rec.get("canonical_smiles")
            tgt = rec.get("target_chembl_id")
            pval = rec.get("pchembl_value")
            if not smi or not tgt or pval is None:
                continue
            if per_target_counts.get(tgt, 0) >= cfg.chembl_pretrain_per_target_cap:
                continue
            per_target_counts[tgt] = per_target_counts.get(tgt, 0) + 1
            rows.append({"smiles": smi, "target_chembl_id": tgt, cfg.chembl_pretrain_endpoint_name: float(pval)})
            if len(rows) >= cfg.chembl_pretrain_max_compounds:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)


def fetch_chembl_pretraining_set_broad(cfg: Config) -> pd.DataFrame:
    '''Original target-agnostic strategy: sample activities across arbitrary human
    targets. Used as the "broad" mode, and as an automatic fallback if domain-targeted
    search doesn't find enough related targets.'''
    from chembl_webresource_client.new_client import new_client

    activity_client = new_client.activity
    rows, seen_targets, per_target_counts = [], set(), {}
    query = activity_client.filter(
        standard_type__in=list(cfg.chembl_pretrain_standard_types),
        target_organism=cfg.chembl_pretrain_organism,
        pchembl_value__isnull=False,
    ).only(["canonical_smiles", "target_chembl_id", "pchembl_value"])

    for rec in tqdm(query, total=cfg.chembl_pretrain_max_compounds * 3, desc="ChEMBL pretraining set (broad)"):
        smi, tgt, pval = rec.get("canonical_smiles"), rec.get("target_chembl_id"), rec.get("pchembl_value")
        if not smi or not tgt or pval is None:
            continue
        if tgt not in per_target_counts and len(seen_targets) >= cfg.chembl_pretrain_max_targets:
            continue
        if per_target_counts.get(tgt, 0) >= 60:
            continue
        seen_targets.add(tgt)
        per_target_counts[tgt] = per_target_counts.get(tgt, 0) + 1
        rows.append({"smiles": smi, "target_chembl_id": tgt, cfg.chembl_pretrain_endpoint_name: float(pval)})
        if len(rows) >= cfg.chembl_pretrain_max_compounds:
            break
    return pd.DataFrame(rows)


def fetch_chembl_pretraining_set(cfg: Config) -> pd.DataFrame:
    cache_path = Path(cfg.chembl_pretrain_cache)
    if cache_path.exists() and not cfg.force_rebuild_chembl_cache:
        print(f"Found cached ChEMBL pretraining set: {cache_path}")
        return pd.read_csv(cache_path)

    try:
        if cfg.chembl_pretrain_mode == "domain_targeted":
            print(f"Searching ChEMBL for targets related to PSMA via {len(cfg.chembl_pretrain_target_keywords)} "
                  f"keyword(s): {list(cfg.chembl_pretrain_target_keywords)}")
            targets_df = find_related_chembl_targets(cfg)
            print(f"Found {len(targets_df):,} matching targets.")
            if len(targets_df):
                print(targets_df[["pref_name", "organism", "matched_keyword"]].to_string(index=False))

            if len(targets_df) < cfg.chembl_pretrain_min_targets_or_fallback_to_broad:
                print(f"Fewer than {cfg.chembl_pretrain_min_targets_or_fallback_to_broad} related targets found "
                      f"-> falling back to 'broad' mode for this run.")
                df = fetch_chembl_pretraining_set_broad(cfg)
            else:
                df = fetch_activities_for_targets(cfg, targets_df["target_chembl_id"].tolist())
                df = df.merge(targets_df[["target_chembl_id", "pref_name", "matched_keyword"]],
                               on="target_chembl_id", how="left")
        else:
            df = fetch_chembl_pretraining_set_broad(cfg)

        if not len(df):
            raise RuntimeError("ChEMBL query returned no usable rows.")

        df = df.drop_duplicates(subset=["smiles"]).reset_index(drop=True)
        print(f"\nFetched {len(df):,} ChEMBL pretraining compounds across "
              f"{df['target_chembl_id'].nunique():,} targets.")
        if "pref_name" in df.columns:
            print("Compounds per target:")
            print(df["pref_name"].value_counts().head(20).to_string())

    except Exception as e:
        print(f"Could not fetch live ChEMBL data ({e}); falling back to a synthetic set.")
        df = _synthetic_chembl_pretrain_set(cfg)

    df.to_csv(cache_path, index=False)
    print(f"Cached ChEMBL pretraining set to {cache_path}")
    return df


pretrain_df = fetch_chembl_pretraining_set(CONFIG)
pretrain_df.head()


## 3. Part A.2 — BindingDB PSMA set (both endpoints: pIC50 + pKi)

Pulls both affinity columns and pivots to a wide, per-compound table with a `pIC50` column,
a `pKi` column and `NaN` where a compound lacks one. Both are fetched because F7 models
both: of 681 compounds, 283 carry only pIC50, 169 carry both, and **229 carry only pKi** —
so a single-task pIC50 model, as F6 was, cannot see a third of the data.

Part A.2b diagnoses the distributions and Part A.2c turns them into masked multi-task class
labels, dropping the pKi values that turn out to be arithmetic transforms of pIC50 rather
than independent measurements.

In [ ]:
def resolve_bindingdb_tsv_url(cfg: Config) -> tuple:
    '''Scrape the BindingDB download page for the CURRENT 'BindingDB_All_<YYYYMM>_tsv.zip'
    filename (BindingDB republishes this monthly, so a hardcoded name goes stale).'''
    download_page = "https://www.bindingdb.org/rwd/bind/chemsearch/marvin/Download.jsp"
    try:
        resp = requests.get(download_page, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resp.raise_for_status()
        m = re.search(r"download_file=(/rwd/bind/downloads/BindingDB_All_\d{6}_tsv\.zip)", resp.text)
        if m:
            file_param = m.group(1)
            zip_name = file_param.rsplit("/", 1)[-1]
            direct_url = f"https://www.bindingdb.org{file_param}"
            print(f"Resolved current BindingDB TSV: {zip_name}")
            return direct_url, zip_name
        print(f"Could not find the current filename on {download_page}; using CONFIG default.")
    except Exception as e:
        print(f"Could not reach {download_page} ({e}); using CONFIG default.")
    return cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name


def _fetch_to_file(url: str, dest: Path) -> str:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; AD-pipeline/1.0)"}
    with requests.get(url, stream=True, timeout=120, headers=headers) as r:
        r.raise_for_status()
        content_type = r.headers.get("content-type", "")
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pbar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                pbar.update(len(chunk))
    return content_type


def _recover_real_download_link(html_path: Path) -> Optional[str]:
    try:
        text = html_path.read_text(errors="ignore")
    except Exception:
        return None
    m = re.search(r'(https://www\.bindingdb\.org/rwd/bind/downloads/[^\s"\x27<>)]+\.zip)', text)
    return m.group(1) if m else None


def download_bindingdb(cfg: Config) -> Optional[Path]:
    '''Download (if needed) and unzip the full BindingDB TSV dump. Returns None (instead of
    raising) if the download can't be completed, so the notebook can fall back to a
    synthetic fine-tuning set and still run end-to-end.'''
    data_dir = Path(cfg.data_dir)
    try:
        if cfg.auto_resolve_bindingdb_url:
            url, zip_name = resolve_bindingdb_tsv_url(cfg)
        else:
            url, zip_name = cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name

        zip_path = data_dir / zip_name
        tsv_path = data_dir / zip_name.replace(".zip", "")

        if tsv_path.exists() and not cfg.force_redownload:
            print(f"Found existing TSV: {tsv_path}")
            return tsv_path

        if not zip_path.exists() or cfg.force_redownload:
            print(f"Downloading {url} ...")
            content_type = _fetch_to_file(url, zip_path)
            print(f"Saved to {zip_path} (content-type: {content_type})")

            if not zipfile.is_zipfile(zip_path):
                real_url = _recover_real_download_link(zip_path)
                if real_url and real_url != url:
                    print(f"Not a zip — retrying with the real link found on the landing page: {real_url}")
                    _fetch_to_file(real_url, zip_path)

        if not zipfile.is_zipfile(zip_path):
            with open(zip_path, "rb") as f:
                head = f.read(300)
            zip_path.unlink(missing_ok=True)
            raise RuntimeError(f"Downloaded file is not a valid zip. First bytes: {head!r}")

        print("Unzipping ...")
        with zipfile.ZipFile(zip_path) as zf:
            member = [m for m in zf.namelist() if m.lower().endswith(".tsv")][0]
            with zf.open(member) as src, open(tsv_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
        print(f"Extracted: {tsv_path}")
        return tsv_path

    except Exception as e:
        print(f"Could not obtain the BindingDB TSV dump ({e}).")
        return None


def query_bindingdb_duckdb_multitask(tsv_path: Path, cfg: Config) -> pd.DataFrame:
    '''Pull one long dataframe with a row per (compound, endpoint) for every column in
    `cfg.affinity_columns`, using DuckDB SQL streaming over the TSV (never materialized
    fully in pandas).'''
    con = duckdb.connect(database=":memory:")
    con.execute("PRAGMA threads=4;")
    con.execute(f'''
        CREATE OR REPLACE VIEW bdb AS
        SELECT * FROM read_csv(
            '{tsv_path.as_posix()}', delim='\t', header=True, quote='"', escape='"',
            strict_mode=False, ignore_errors=True, all_varchar=True
        )
    ''')

    base_where = ['"Ligand SMILES" IS NOT NULL']
    if cfg.target_uniprot_id:
        uniprot_conditions = [
            f'"UniProt (SwissProt) Primary ID of Target Chain {i}" = \'{cfg.target_uniprot_id}\''
            for i in range(1, 6)
        ]
        base_where.append("(" + " OR ".join(uniprot_conditions) + ")")
    if cfg.target_name_like:
        safe = cfg.target_name_like.replace("'", "''")
        base_where.append(f'"Target Name" ILIKE \'%{safe}%\'')
    if cfg.organism_like:
        safe = cfg.organism_like.replace("'", "''")
        base_where.append(f'"Target Source Organism According to Curator or DataSource" ILIKE \'%{safe}%\'')

    frames = []
    for col, endpoint in zip(cfg.affinity_columns, cfg.endpoint_names):
        where_sql = " AND ".join(base_where + [f'"{col}" IS NOT NULL'])
        query = f'''
            SELECT
                "Ligand SMILES"                                    AS smiles,
                "BindingDB Ligand Name"                            AS ligand_name,
                "{col}"                                            AS affinity_raw,
                '{endpoint}'                                       AS endpoint_name
            FROM bdb
            WHERE {where_sql}
        '''
        part = con.execute(query).fetch_df()
        print(f"  {endpoint} ('{col}'): {len(part):,} raw rows.")
        frames.append(part)
    con.close()
    return pd.concat(frames, ignore_index=True)


def _synthetic_bindingdb_finetune_set(cfg: Config, n: int = 250) -> pd.DataFrame:
    print("!! Using SYNTHETIC BindingDB PSMA fine-tuning data (network/download unavailable) !!")
    demo_smiles = [
        "CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
        "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
        "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
        "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12",
        "NC(CCC(=O)O)C(=O)O", "OC(=O)CCC(NC(=O)NC(CCC(=O)O)C(=O)O)C(=O)O",
    ]
    rng = np.random.RandomState(cfg.random_state)
    rows = []
    for i in range(n):
        smi = demo_smiles[i % len(demo_smiles)]
        row = {"smiles": smi, "ligand_name": f"synthetic_{i}"}
        if rng.rand() < 0.75:
            row[cfg.endpoint_names[0]] = float(np.clip(rng.normal(6.8, 1.2), 3, 10))
        if rng.rand() < 0.6:
            row[cfg.endpoint_names[1]] = float(np.clip(rng.normal(6.5, 1.3), 3, 10))
        rows.append(row)
    return pd.DataFrame(rows)


def clean_bindingdb_multitask(df_long: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    '''Coerce to numeric, drop censored (>,<) measurements, compute pAffinity, take the
    per-compound median for each endpoint separately, then pivot wide so every
    compound is one row with (possibly-NaN) pIC50 / pKi columns.'''
    df = df_long.copy()
    df["is_censored"] = df["affinity_raw"].astype(str).str.contains("[<>]", regex=True)
    df = df[~df["is_censored"]]
    df["affinity_nm"] = pd.to_numeric(df["affinity_raw"], errors="coerce")
    df = df.dropna(subset=["affinity_nm", "smiles"])
    df = df[(df["affinity_nm"] >= cfg.min_affinity_nm) & (df["affinity_nm"] <= cfg.max_affinity_nm)]
    df["pAffinity"] = 9 - np.log10(df["affinity_nm"])

    per_endpoint = (
        df.groupby(["smiles", "endpoint_name"], as_index=False)
        .agg({"pAffinity": "median", "ligand_name": "first"})
    )
    wide = per_endpoint.pivot_table(index="smiles", columns="endpoint_name", values="pAffinity")
    names = per_endpoint.groupby("smiles")["ligand_name"].first()
    wide = wide.join(names).reset_index()
    for ep in cfg.endpoint_names:
        if ep not in wide.columns:
            wide[ep] = np.nan
    print(f"Cleaned multi-task fine-tuning set: {len(wide):,} unique compounds "
          + ", ".join(f"({ep}: {wide[ep].notna().sum():,})" for ep in cfg.endpoint_names))
    return wide.reset_index(drop=True)


# --- run it -------------------------------------------------------------------
tsv_path = download_bindingdb(CONFIG)
if tsv_path is not None:
    raw_bdb_df = query_bindingdb_duckdb_multitask(tsv_path, CONFIG)
    train_df = clean_bindingdb_multitask(raw_bdb_df, CONFIG)
else:
    train_df = _synthetic_bindingdb_finetune_set(CONFIG)

train_df.head()


## 3b. Part A.2b — Where should the activity-class cuts go?

A read-only diagnostic, added to settle the threshold question before any switch from
numeric prediction to activity classes. It defines nothing the rest of the pipeline
consumes, so it can be run or skipped freely.

It needs `raw_bdb_df` and `train_df`, both created by the cell above — `raw_bdb_df`
is the **pre-cleaning** long frame, so it still contains the censored (`>` / `<`)
measurements that `clean_bindingdb_multitask` discards. That is what makes the
censored-row audit possible without re-downloading anything. If the kernel has been
restarted, re-run the cell above first (it reuses the cached TSV).

Four questions, in order:

1. **Shape.** Percentiles, skew, kurtosis, a Gaussian-mixture BIC comparison for
   1 / 2 / 3 components, and a kernel-density scan for the *valleys* — the emptiest
   places to put a boundary, where a cut splits the fewest near-identical compounds.
2. **Censored-row audit.** How many `>` and `<` rows were dropped per endpoint, where
   their bounds sit (a `>1 µM` is a far weaker claim than a `>100 µM`), and how many
   distinct compounds they would add back as confident inactives or actives. This is
   the single number that decides three-class versus binary.
3. **Class counts** under candidate schemes side by side: field-standard
   (100 nM / 10 µM), PSMA-tightened (10 nM / 1 µM), PSMA-strict (1 nM / 1 µM), plus
   observed tertiles and the density valleys for comparison — each shown before and
   after the censored data is folded in.
4. **Boundary noise.** For compounds measured both ways, how often pIC50 and pKi land
   in *different* classes. That disagreement is assay noise on the same molecule, so it
   is a hard ceiling on classifier accuracy. A scheme below roughly 80% agreement has
   boundaries cutting through noise rather than signal.

Note on interpretation: anchor the thresholds on pharmacology and use the distribution
to *check* them — that each class has enough members, and that the cut does not land on
a density peak. Choosing tertiles because they balance the classes makes "active" mean
"top third of this particular snapshot", which is not transferable and shifts the moment
ChEMBL data or decoys are added.


In [ ]:
# =====================================================================================
# Part A.2b — Label-distribution diagnostic: where should the activity-class cuts go?
#
# Drop this in as a new cell immediately AFTER cell 9 (the BindingDB pull), which is
# where `raw_bdb_df` and `train_df` come into existence. It is read-only: it adds no
# variables the rest of the pipeline depends on and changes nothing.
#
# It answers four questions:
#   1. What shape is the label distribution really? (percentiles, skew, bimodality)
#   2. How many censored (">" / "<") rows were dropped, and what would they add back
#      as confident inactives / actives?
#   3. How many compounds land in each class under candidate threshold schemes?
#   4. How noisy is the labelling at the boundaries, measured by pIC50-vs-pKi
#      disagreement on compounds that have both?
# =====================================================================================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

EPS = list(CONFIG.endpoint_names)          # ['pIC50', 'pKi']
_COLS = dict(zip(CONFIG.endpoint_names, CONFIG.affinity_columns))

print("=" * 86)
print("1. SHAPE OF THE LABEL DISTRIBUTION")
print("=" * 86)

_shape_rows = []
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) < 10:
        continue
    q = np.percentile(v, [1, 5, 10, 25, 50, 75, 90, 95, 99])
    _shape_rows.append({
        "endpoint": ep, "n": len(v),
        "mean": v.mean(), "sd": v.std(ddof=0),
        "p1": q[0], "p5": q[1], "p10": q[2], "p25": q[3], "p50": q[4],
        "p75": q[5], "p90": q[6], "p95": q[7], "p99": q[8],
        "skew": stats.skew(v), "excess_kurt": stats.kurtosis(v),
        "min": v.min(), "max": v.max(),
    })
shape_df = pd.DataFrame(_shape_rows).set_index("endpoint").round(3)
display(shape_df)

# --- is it one mode or several? -------------------------------------------------------
# A kernel-density scan: local minima of the density between the 5th and 95th percentile
# are the natural "valleys", i.e. the least-populated places to put a cut.
def density_valleys(v, n_grid=512, lo_pct=5, hi_pct=95):
    kde = stats.gaussian_kde(v)
    lo, hi = np.percentile(v, [lo_pct, hi_pct])
    grid = np.linspace(lo, hi, n_grid)
    dens = kde(grid)
    # interior local minima
    is_min = (dens[1:-1] < dens[:-2]) & (dens[1:-1] < dens[2:])
    valleys = grid[1:-1][is_min]
    depths = dens[1:-1][is_min]
    order = np.argsort(depths)              # shallowest density = emptiest = best cut
    return grid, dens, valleys[order], depths[order]

print("\nBimodality check and natural density valleys")
print("-" * 86)
valley_store = {}
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) < 30:
        print(f"  {ep}: too few labelled rows for a density scan.")
        continue
    grid, dens, valleys, depths = density_valleys(v)
    valley_store[ep] = (grid, dens, valleys)
    # Crude unimodality signal: compare the best single Gaussian to a 2-component mix.
    try:
        from sklearn.mixture import GaussianMixture
        X = v.reshape(-1, 1)
        bic1 = GaussianMixture(1, random_state=0).fit(X).bic(X)
        bic2 = GaussianMixture(2, random_state=0).fit(X).bic(X)
        bic3 = GaussianMixture(3, random_state=0).fit(X).bic(X)
        best = {1: bic1, 2: bic2, 3: bic3}
        n_modes = min(best, key=best.get)
        verdict = (f"{n_modes}-component mixture fits best "
                   f"(BIC 1/2/3 = {bic1:.0f} / {bic2:.0f} / {bic3:.0f})")
    except Exception as e:
        verdict = f"mixture fit unavailable ({e})"
    vstr = ", ".join(f"{x:.2f}" for x in valleys[:4]) if len(valleys) else "none found"
    print(f"  {ep}: {verdict}")
    print(f"         emptiest cut points (shallowest density first): {vstr}")
    print(f"         -> a threshold placed in a valley splits fewer near-identical "
          f"compounds across the boundary")

print("\n" + "=" * 86)
print("2. CENSORED-ROW AUDIT  (the data classification would recover)")
print("=" * 86)

_cens_rows = []
_recovered = {}          # endpoint -> DataFrame of censored measurements
for ep in EPS:
    col = _COLS[ep]
    sub = raw_bdb_df[raw_bdb_df["endpoint_name"] == ep].copy()
    raw = sub["affinity_raw"].astype(str)
    sub["qualifier"] = raw.str.extract(r"^\s*([<>]=?)")[0]
    sub["value_nm"] = pd.to_numeric(raw.str.replace(r"^\s*[<>]=?\s*", "", regex=True),
                                    errors="coerce")
    cens = sub[sub["qualifier"].notna() & sub["value_nm"].notna()].copy()
    cens["pAffinity_bound"] = 9 - np.log10(cens["value_nm"])
    _recovered[ep] = cens

    greater = cens[cens["qualifier"].str.startswith(">")]
    lesser = cens[cens["qualifier"].str.startswith("<")]
    _cens_rows.append({
        "endpoint": ep,
        "raw_rows": len(sub),
        "censored_rows": len(cens),
        "gt_rows_INACTIVE": len(greater),
        "lt_rows_ACTIVE": len(lesser),
        "gt_distinct_cpds": greater["smiles"].nunique(),
        "lt_distinct_cpds": lesser["smiles"].nunique(),
        "numeric_cpds_now": int(train_df[ep].notna().sum()),
    })

cens_df = pd.DataFrame(_cens_rows).set_index("endpoint")
display(cens_df)

print("Where the '>' bounds sit (these become inactives; a '>1 uM' is a much weaker")
print("statement than a '>100 uM', so they are not interchangeable):")
for ep in EPS:
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g) == 0:
        print(f"  {ep}: no '>' rows.")
        continue
    vc = g["value_nm"].value_counts().sort_index()
    print(f"  {ep}:")
    for val, n in vc.items():
        print(f"      > {val:>12,.1f} nM  (pAffinity < {9 - np.log10(val):.2f})   n = {n}")

# How many of the '>' compounds are genuinely NEW, vs already present with a number?
print("\nOverlap with the compounds already in train_df:")
for ep in EPS:
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g) == 0:
        continue
    new = set(g["smiles"]) - set(train_df["smiles"])
    print(f"  {ep}: {g['smiles'].nunique()} distinct '>' compounds, "
          f"of which {len(new)} are not in the current training set at all")
print("  (Note: matching is on raw SMILES strings, as the pipeline does, so this is an")
print("   under-count of true overlap - standardisation would merge more. Dedup on")
print("   InChIKey after standardisation before trusting these numbers.)")

print("\n" + "=" * 86)
print("3. CLASS COUNTS UNDER CANDIDATE THRESHOLD SCHEMES")
print("=" * 86)

# (name, active_cut, inactive_cut) on the pAffinity scale; class is
#   ACTIVE    if  p >= active_cut
#   INACTIVE  if  p <  inactive_cut
#   MODERATE  otherwise
_p = lambda nm: 9 - np.log10(nm)

SCHEMES = {
    "field-standard (100 nM / 10 uM)": (_p(100), _p(10_000)),
    "PSMA-tightened (10 nM / 1 uM)":   (_p(10),  _p(1_000)),
    "PSMA-strict (1 nM / 1 uM)":       (_p(1),   _p(1_000)),
}
# add tertiles of the observed numeric distribution, per endpoint, for comparison only
for ep in EPS:
    v = train_df[ep].dropna().values
    if len(v) > 30:
        lo, hi = np.percentile(v, [33.3, 66.7])
        SCHEMES[f"tertiles of observed {ep} ({hi:.2f} / {lo:.2f})"] = (hi, lo)
# add the emptiest density valleys, if two were found
for ep in EPS:
    if ep in valley_store and len(valley_store[ep][2]) >= 2:
        two = sorted(valley_store[ep][2][:2])
        SCHEMES[f"density valleys of {ep} ({two[1]:.2f} / {two[0]:.2f})"] = (two[1], two[0])


def classify(p, active_cut, inactive_cut):
    if not np.isfinite(p):
        return np.nan
    if p >= active_cut:
        return "active"
    if p < inactive_cut:
        return "inactive"
    return "moderate"


rows = []
for name, (a_cut, i_cut) in SCHEMES.items():
    for ep in EPS:
        v = train_df[ep].dropna()
        cls = v.map(lambda p: classify(p, a_cut, i_cut))
        counts = cls.value_counts()
        # censored '>' rows become inactives, '<' rows become actives
        g = _recovered[ep]
        add_inact = g[g["qualifier"].str.startswith(">")]
        # only count a '>' row as inactive if its bound actually implies inactivity
        add_inact = add_inact[add_inact["pAffinity_bound"] <= i_cut]
        add_act = g[g["qualifier"].str.startswith("<")]
        add_act = add_act[add_act["pAffinity_bound"] >= a_cut]
        rows.append({
            "scheme": name, "endpoint": ep,
            "active": int(counts.get("active", 0)),
            "moderate": int(counts.get("moderate", 0)),
            "inactive": int(counts.get("inactive", 0)),
            "add_inact": add_inact["smiles"].nunique(),
            "add_act": add_act["smiles"].nunique(),
            "inact_total": int(counts.get("inactive", 0)) + add_inact["smiles"].nunique(),
        })

class_counts = pd.DataFrame(rows).set_index(["scheme", "endpoint"])
display(class_counts)

print("Reading this table:")
print("  * 'inact_total' is the figure that decides 3-class vs binary. Under ~80")
print("    compounds and three classes will not train reliably - go binary")
print("    (active vs not-active) instead.")
print("  * A scheme that puts >70% of compounds in one class will need class weights")
print("    or stratified sampling whichever way you go.")
print("  * Prefer a pharmacologically meaningful cut that happens to sit near a density")
print("    valley. Do NOT pick tertiles because they balance the classes - that makes")
print("    'active' mean 'top third of this snapshot', which moves when data is added.")

print("\n" + "=" * 86)
print("4. BOUNDARY NOISE: do pIC50 and pKi agree on the class?")
print("=" * 86)
print("Compounds measured both ways should land in the same class. Where they do not,")
print("the disagreement is assay noise, and it bounds how good any classifier can be.\n")

both = train_df[train_df[EPS[0]].notna() & train_df[EPS[1]].notna()]
print(f"Compounds with both {EPS[0]} and {EPS[1]}: {len(both)}")
if len(both) >= 10:
    d = (both[EPS[0]] - both[EPS[1]]).abs()
    print(f"  |{EPS[0]} - {EPS[1]}|:  median {d.median():.2f}, mean {d.mean():.2f}, "
          f"90th pct {d.quantile(0.9):.2f}, max {d.max():.2f} log units")
    print(f"  fraction differing by >0.5 log: {100 * (d > 0.5).mean():.1f}%")
    print(f"  fraction differing by >1.0 log: {100 * (d > 1.0).mean():.1f}%")
    print(f"  Pearson r between the two endpoints: {both[EPS[0]].corr(both[EPS[1]]):.3f}\n")
    for name, (a_cut, i_cut) in SCHEMES.items():
        c0 = both[EPS[0]].map(lambda p: classify(p, a_cut, i_cut))
        c1 = both[EPS[1]].map(lambda p: classify(p, a_cut, i_cut))
        agree = (c0 == c1).mean()
        try:
            from sklearn.metrics import cohen_kappa_score
            kappa = cohen_kappa_score(c0, c1)
            kstr = f", kappa = {kappa:.3f}"
        except Exception:
            kstr = ""
        print(f"  {name:<46s} agreement = {100 * agree:5.1f}%{kstr}")
    print("\n  Agreement here is a CEILING on classifier accuracy: the same molecule,")
    print("  measured twice, disagrees this often. If a scheme falls below ~80%, its")
    print("  boundaries are cutting through the noise rather than through the signal.")
else:
    print("  Too few dual-measured compounds to assess boundary noise.")

print("\n" + "=" * 86)
print("5. PLOTS")
print("=" * 86)

fig, axes = plt.subplots(1, len(EPS), figsize=(7.5 * len(EPS), 4.8))
axes = np.atleast_1d(axes)
_cut_colors = plt.get_cmap("tab10")
for ax, ep in zip(axes, EPS):
    v = train_df[ep].dropna().values
    ax.hist(v, bins=40, color="#9aa0a6", edgecolor="white", linewidth=0.5,
            alpha=0.85, label=f"{ep} (n={len(v)})")
    if ep in valley_store:
        grid, dens, valleys = valley_store[ep]
        ax2 = ax.twinx()
        ax2.plot(grid, dens, color="#1a1a1a", linewidth=1.6, label="density")
        ax2.set_yticks([])
        for x in valleys[:3]:
            ax2.axvline(x, color="#1a1a1a", linestyle=":", linewidth=1.2, alpha=0.7)
    for ci, (name, (a_cut, i_cut)) in enumerate(list(SCHEMES.items())[:3]):
        col = _cut_colors(ci)
        ax.axvline(a_cut, color=col, linestyle="--", linewidth=1.8,
                   label=f"{name.split(' (')[0]}: active")
        ax.axvline(i_cut, color=col, linestyle="-.", linewidth=1.4, alpha=0.8)
    g = _recovered[ep]
    g = g[g["qualifier"].str.startswith(">")]
    if len(g):
        ax.scatter(g["pAffinity_bound"], np.full(len(g), ax.get_ylim()[1] * 0.04),
                   marker="|", s=180, color="#d1495b", alpha=0.6,
                   label=f"censored '>' bounds (n={len(g)})")
    ax.set_xlabel(ep)
    ax.set_ylabel("compounds")
    ax.set_title(f"{ep}: distribution, density valleys (dotted) and candidate cuts")
    ax.legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()

print("Dotted vertical lines are density valleys - the emptiest places to cut.")
print("Red ticks are the censored '>' bounds that classification would recover as")
print("inactives; where they pile up tells you where the real inactive mass sits.")

## 3c. Part A.2c — Multi-task classification labels

Four steps, in order.

**Find the derived values.** `detect_derived_endpoint` looks for blocks of compounds whose
two endpoints differ by a *constant*. A fixed offset shared across many compounds from one
source is the signature of one endpoint being computed from the other — Cheng–Prusoff
Ki = IC50 / (1 + [S]/Km) with a single substrate concentration produces exactly this — and
not of two independent assays. F6's version reported overlapping rounding windows as
separate blocks (three rows for one block of 111); this one merges blocks whose offsets sit
within `derived_offset_tol` of each other, so each real block is reported once.

**Drop them.** For a compound in a derived block, the pKi label goes and the pIC50 label
stays. This is not a judgement call about data quality — a value equal to pIC50 ÷ 140 adds
no information by construction, and keeping it would make a single pKi threshold wrong for
most of the column.

**Recover the censored measurements, on both endpoints.** A `>` bound at or below the
active cut proves the compound is not active and becomes a negative; a `<` bound at or
above it becomes a positive. Bounds that straddle the cut are uninformative and are
dropped with a count. This is the only step in the whole pipeline that adds genuinely new
information rather than rearranging what was there.

**Build the masked label matrix.** One row per compound, one column per endpoint, `NaN`
where that compound has no usable label for that endpoint. Downstream, the network trains
on both columns at once with a masked loss so partially-labelled compounds still
contribute, and the tree and kernel baselines are fitted once per endpoint on that
endpoint's labelled rows.

Each compound also carries a `source_tag` parsed from its BindingDB ligand name, which is
what makes single-filing over-representation visible and lets the leave-one-cluster-out
report in Part E.2 name the family it is holding out.

In [ ]:
# --- Multi-task label assembly ---------------------------------------------------------
import re

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

EPS = list(CONFIG.modelled_endpoints)
EP_COLS = dict(zip(CONFIG.endpoint_names, CONFIG.affinity_columns))
A_CUTS = dict(CONFIG.active_thresholds)
CLASS_NAMES = ["not-active", "active"]
POSITIVE_CLASS = 1
N_CLASSES = 2

GENERIC_SOURCE_TAGS = {"CHEMBL", "other", "unknown"}
_PATENT_RE = re.compile(r"\b((?:US|WO|EP|CN|JP|KR)\s?\d{6,})", re.I)
_CHEMBL_RE = re.compile(r"\b(CHEMBL\d+)", re.I)


def source_tag(name) -> str:
    """Coarse provenance from the BindingDB ligand name. A patent number identifies one
    filing; CHEMBL and 'other' are catch-all buckets and must not be read as one source."""
    if not isinstance(name, str) or not name.strip():
        return "unknown"
    m = _PATENT_RE.search(name)
    if m:
        return m.group(1).replace(" ", "").upper()
    if _CHEMBL_RE.search(name):
        return "CHEMBL"
    return "other"


print(f"Modelling {len(EPS)} endpoints jointly: {EPS}")
for ep in EPS:
    print(f"  {ep}: active >= {A_CUTS[ep]:.2f}  "
          f"(<= {10 ** (9 - A_CUTS[ep]):.3g} nM)")


# =====================================================================================
# 1. Derived-value detection, with overlapping blocks merged
# =====================================================================================
def detect_derived_endpoint(df_wide, ep_a, ep_b, min_block=5, tol=0.02):
    """Blocks of compounds whose two endpoints differ by a FIXED amount.

    Returns (report, mask). Unlike F6's version, candidate offsets whose means fall within
    `tol` of an already-accepted block are merged into it rather than reported separately —
    F6 printed one block of 111 compounds as three rows of 106/103/102 because its +/-tol
    windows around the 2.14 / 2.15 / 2.16 rounding buckets overlapped.
    """
    empty = (pd.DataFrame(), pd.Series(False, index=df_wide.index))
    if ep_a not in df_wide.columns or ep_b not in df_wide.columns:
        return empty
    both = df_wide[df_wide[ep_a].notna() & df_wide[ep_b].notna()].copy()
    if len(both) < min_block:
        return empty
    both["_diff"] = both[ep_b] - both[ep_a]

    blocks, claimed = [], set()
    # work through candidate offsets most-populated first, skipping compounds already
    # claimed by a wider block, so each compound belongs to exactly one block
    for val, _n in both["_diff"].round(2).value_counts().items():
        sel = both[((both["_diff"] - val).abs() <= tol) & (~both.index.isin(claimed))]
        if len(sel) < min_block:
            continue
        if any(abs(b["offset"] - float(sel["_diff"].mean())) <= tol * 1.5 for b in blocks):
            continue                      # same block, different rounding bucket
        srcs = (sel["source_tag"].value_counts() if "source_tag" in sel
                else pd.Series(dtype=int))
        blocks.append({
            "offset": round(float(sel["_diff"].mean()), 4),
            "implied_ratio": round(float(10 ** sel["_diff"].mean()), 2),
            "n_compounds": len(sel),
            "offset_sd": round(float(sel["_diff"].std(ddof=0)), 5),
            "dominant_source": srcs.index[0] if len(srcs) else "?",
            "source_share": round(float(srcs.iloc[0] / len(sel)), 3) if len(srcs) else np.nan,
            "_idx": list(sel.index),
        })
        claimed.update(sel.index)

    mask = pd.Series(False, index=df_wide.index)
    mask.loc[sorted(claimed)] = True
    report = (pd.DataFrame([{k: v for k, v in b.items() if k != "_idx"} for b in blocks])
              .sort_values("n_compounds", ascending=False) if blocks else pd.DataFrame())
    return report, mask


train_df_mt = train_df.copy()
train_df_mt["source_tag"] = train_df_mt["ligand_name"].map(source_tag)
derived_report, derived_mask = detect_derived_endpoint(
    train_df_mt, CONFIG.endpoint_names[0], CONFIG.endpoint_names[1],
    min_block=CONFIG.derived_min_block_size, tol=CONFIG.derived_offset_tol,
)
derived_smiles = set(train_df_mt.loc[derived_mask, "smiles"])

print("\n" + "=" * 90)
print("DERIVED-VALUE DETECTOR — fixed-offset blocks between the two endpoints")
print("=" * 90)
if len(derived_report):
    display(derived_report)
    _n_pairs = int((train_df_mt[CONFIG.endpoint_names[0]].notna()
                    & train_df_mt[CONFIG.endpoint_names[1]].notna()).sum())
    print(f"{int(derived_mask.sum())} of {_n_pairs} dual-measured compounds "
          f"({100 * derived_mask.sum() / max(_n_pairs, 1):.0f}%) sit in a fixed-offset block.")
    print("A constant offset across many compounds from one source means the second endpoint")
    print("was computed from the first (Cheng-Prusoff), not measured. Those pKi labels carry")
    print("no information and would make any single pKi threshold wrong for most of the")
    if CONFIG.drop_derived_endpoint_values:
        print("column, so they are DROPPED below. Each affected compound keeps its pIC50.")
    else:
        print("column. drop_derived_endpoint_values is False, so they are KEPT — expect the")
        print("endpoint-agreement problem from Part A.2b to persist.")
else:
    print("No fixed-offset blocks found. The endpoints look independently measured.")


# =====================================================================================
# 2. Per-endpoint re-derivation from the raw pull, keeping censored rows
# =====================================================================================
def build_endpoint_labels(ep: str, a_cut: float):
    """Returns {smiles: record} for one endpoint, with numeric and censored labels."""
    col = EP_COLS[ep]
    sub = raw_bdb_df[raw_bdb_df["endpoint_name"] == ep].copy()
    raw = sub["affinity_raw"].astype(str)
    sub["qualifier"] = raw.str.extract(r"^\s*([<>]=?)")[0]
    sub["value_nm"] = pd.to_numeric(raw.str.replace(r"^\s*[<>]=?\s*", "", regex=True),
                                    errors="coerce")
    sub = sub.dropna(subset=["value_nm", "smiles"])
    sub = sub[(sub["value_nm"] >= CONFIG.min_affinity_nm)
              & (sub["value_nm"] <= CONFIG.max_affinity_nm)]
    sub["pAff"] = 9 - np.log10(sub["value_nm"])

    numeric = sub[sub["qualifier"].isna()]
    gt = sub[sub["qualifier"].notna() & sub["qualifier"].str.startswith(">")]
    lt = sub[sub["qualifier"].notna() & sub["qualifier"].str.startswith("<")]

    num_agg = (numeric.groupby("smiles", as_index=False)
               .agg(pAff=("pAff", "median"), ligand_name=("ligand_name", "first"),
                    n_meas=("pAff", "size")))
    # several '>' bounds must all hold -> the most informative is the smallest
    gt_agg = (gt.groupby("smiles", as_index=False)
              .agg(p_upper=("pAff", "min"), ligand_name=("ligand_name", "first")))
    lt_agg = (lt.groupby("smiles", as_index=False)
              .agg(p_lower=("pAff", "max"), ligand_name=("ligand_name", "first")))

    rec, stats = {}, {"numeric_rows": len(numeric), "numeric_cpds": len(num_agg),
                      "gt_rows": len(gt), "lt_rows": len(lt),
                      "recovered_inactive": 0, "recovered_active": 0,
                      "dropped_ambiguous": 0, "dropped_derived": 0}

    for r in num_agg.itertuples(index=False):
        if (CONFIG.drop_derived_endpoint_values and ep != CONFIG.primary_endpoint
                and r.smiles in derived_smiles):
            stats["dropped_derived"] += 1
            continue
        rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": float(r.pAff),
                         "y": int(r.pAff >= a_cut), "label_source": "numeric",
                         "n_meas": int(r.n_meas)}

    if CONFIG.keep_censored_as_labels:
        for r in gt_agg.itertuples(index=False):
            if r.smiles in rec:
                continue
            if (CONFIG.drop_derived_endpoint_values and ep != CONFIG.primary_endpoint
                    and r.smiles in derived_smiles):
                stats["dropped_derived"] += 1
                continue
            if r.p_upper <= a_cut:                      # provably not active
                rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": np.nan, "y": 0,
                                 "label_source": "censored", "n_meas": 0}
                stats["recovered_inactive"] += 1
            else:
                stats["dropped_ambiguous"] += 1
        for r in lt_agg.itertuples(index=False):
            if r.smiles in rec:
                continue
            if r.p_lower >= a_cut:                      # provably active
                rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": np.nan, "y": 1,
                                 "label_source": "censored", "n_meas": 0}
                stats["recovered_active"] += 1
            else:
                stats["dropped_ambiguous"] += 1
    return rec, stats


endpoint_records, endpoint_stats = {}, {}
for ep in EPS:
    endpoint_records[ep], endpoint_stats[ep] = build_endpoint_labels(ep, A_CUTS[ep])

print("\n" + "=" * 90)
print("PER-ENDPOINT LABEL ASSEMBLY")
print("=" * 90)
display(pd.DataFrame(endpoint_stats).T)


# =====================================================================================
# 3. Masked multi-task table
# =====================================================================================
all_smiles = sorted(set().union(*[set(r) for r in endpoint_records.values()]))
rows = []
for smi in all_smiles:
    row = {"smiles": smi}
    name = None
    for ep in EPS:
        r = endpoint_records[ep].get(smi)
        if r is None:
            row[ep] = np.nan
            row[f"y_{ep}"] = np.nan
            row[f"src_{ep}"] = None
        else:
            row[ep] = r["pAff"]
            row[f"y_{ep}"] = float(r["y"])
            row[f"src_{ep}"] = r["label_source"]
            name = name or r["ligand_name"]
    row["ligand_name"] = name
    rows.append(row)

mt_df = pd.DataFrame(rows)
mt_df["source_tag"] = mt_df["ligand_name"].map(source_tag)
mt_df["n_labels"] = mt_df[[f"y_{ep}" for ep in EPS]].notna().sum(axis=1)

print("\n" + "=" * 90)
print("MULTI-TASK LABEL COVERAGE")
print("=" * 90)
_cov = []
for ep in EPS:
    col = f"y_{ep}"
    _cov.append({"endpoint": ep, "labelled": int(mt_df[col].notna().sum()),
                 "active": int((mt_df[col] == 1).sum()),
                 "not_active": int((mt_df[col] == 0).sum()),
                 "base_rate": round(float(mt_df.loc[mt_df[col].notna(), col].mean()), 3)})
display(pd.DataFrame(_cov).set_index("endpoint"))

_only = {ep: int(((mt_df[f"y_{ep}"].notna()) & (mt_df["n_labels"] == 1)).sum())
         for ep in EPS}
print(f"Compounds with BOTH labels : {int((mt_df['n_labels'] == 2).sum())}")
for ep, n in _only.items():
    print(f"Compounds with {ep} only{' ' * max(0, 7 - len(ep))}: {n}")
print(f"TOTAL modelled compounds   : {len(mt_df):,}")
print(f"\nF6 modelled {CONFIG.primary_endpoint} alone and so used "
      f"{int(mt_df[f'y_{CONFIG.primary_endpoint}'].notna().sum()):,} compounds. "
      f"Multi-task reaches {len(mt_df):,} "
      f"(+{100 * len(mt_df) / max(int(mt_df[f'y_{CONFIG.primary_endpoint}'].notna().sum()), 1) - 100:.0f}%).")

print("\nSource composition:")
_src = mt_df["source_tag"].value_counts()
for tag, n in _src.head(8).items():
    print(f"  {tag:>16s}: {n:4d} ({100 * n / len(mt_df):5.1f}%)")
_specific = _src[~_src.index.isin(GENERIC_SOURCE_TAGS)]
if len(_specific) and _specific.iloc[0] / len(mt_df) > 0.15:
    print(f"\n!! '{_specific.index[0]}' — a single filing — supplies {_specific.iloc[0]} of "
          f"{len(mt_df)} compounds ({100 * _specific.iloc[0] / len(mt_df):.0f}%).")
    print("   Part E.2 holds it out as its own fold so its contribution is measurable.")

train_df = mt_df.copy()
print(f"\n`train_df` is now the multi-task classification set: {train_df.shape}")
train_df.head()

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully inside Colab.")
except ModuleNotFoundError:
    print("Running on a local environment; skipping Google Drive mount as it is not needed.")

## 4. Part A.3 — Curated screening library

Unchanged from the original pipeline: the compounds you want predictions +
applicability-domain flags for. Supports CSV (with a SMILES column) or SDF.


In [ ]:
def load_screening_library(cfg: Config) -> pd.DataFrame:
    # Sanitize path: if it starts with an absolute root slash like '/Enamine_screen_library.sdf',
    # convert it to a local relative path './Enamine_screen_library.sdf' to avoid Windows permission errors.
    raw_path = cfg.screening_library_path
    if raw_path.startswith("/") or raw_path.startswith("\\"):
        raw_path = "." + raw_path

    path = Path(raw_path)
    if not path.exists():
        print(f"Screening library not found at {path}. Generating a demo screening library locally.")
        path.parent.mkdir(parents=True, exist_ok=True)
        demo = pd.DataFrame({
            "compound_id": [f"demo_{i}" for i in range(8)],
            "SMILES": [
                "CCOc1ccc2nc(S(N)(=O)=O)sc2c1",
                "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
                "O=C(O)c1ccccc1O",
                "CC(=O)Oc1ccccc1C(=O)O",
                "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
                "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
                "CN1CCC[C@H]1c1cccnc1",
                "CC(C)NCC(O)COc1cccc2ccccc12"
            ]
        })
        demo.to_csv(path, index=False)
        print(f"Demo screening library saved to: {path}")

    if path.suffix.lower() in (".sdf", ".sd"):
        suppl = Chem.SDMolSupplier(str(path))
        rows = []
        for i, mol in enumerate(suppl):
            if mol is None:
                continue
            smi = Chem.MolToSmiles(mol)
            cid = mol.GetProp("_Name") if mol.HasProp("_Name") else f"SDF_mol_{i}"
            rows.append({"smiles": smi, "compound_id": cid})
        return pd.DataFrame(rows)
    else:
        df = pd.read_csv(path)
        df = df.rename(columns={cfg.screening_smiles_col: "smiles"})
        if cfg.screening_id_col and cfg.screening_id_col in df.columns:
            df = df.rename(columns={cfg.screening_id_col: "compound_id"})
        elif "compound_id" not in df.columns:
            df["compound_id"] = [f"ROW_{i}" for i in range(len(df))]
        return df[["smiles", "compound_id"]]

screen_df = load_screening_library(CONFIG)
screen_df.head()

## 5. Part B.1 — Featurization

Same RDKit standardization + Morgan-fingerprint + descriptor wrapper as the original
pipeline, now applied to **three** sets: the broad ChEMBL pretraining set, the PSMA
BindingDB fine-tuning set, and the screening library. Using the identical featurizer
everywhere is what makes the pretrained trunk's weights meaningful once loaded into
the fine-tuning model — the input feature space has to match.


In [ ]:
class RDKitFeaturizer:
    '''Reusable RDKit wrapper: standardization + fingerprints + descriptors.

    Uses RDKit's full 2D descriptor catalog (~200 physicochemical/topological/graph
    descriptors from `Descriptors._descList`, in the spirit of AutoQSAR's ~500-descriptor
    pool[1]) rather than a small hand-picked subset. The redundancy this introduces is
    dealt with explicitly in Part B.1b below, following AutoQSAR's own near-constant +
    correlation-clustering filter, rather than left for the model to sort out.

    [1] Dixon et al., "AutoQSAR: an automated machine learning tool for best-practice
        QSAR modeling," Future Med. Chem. 2016.
    '''

    DESCRIPTOR_NAMES = [name for name, _ in Descriptors._descList]

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self._normalizer = rdMolStandardize.Normalizer()
        self._lfc = rdMolStandardize.LargestFragmentChooser()
        self._uncharger = rdMolStandardize.Uncharger()
        self._desc_fns = dict(Descriptors._descList)

    def standardize(self, smiles: str) -> Optional[Chem.Mol]:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        try:
            mol = self._normalizer.normalize(mol)
            mol = self._lfc.choose(mol)
            mol = self._uncharger.uncharge(mol)
            Chem.SanitizeMol(mol)
        except Exception:
            return None
        return mol

    def morgan_fp(self, mol: Chem.Mol):
        return AllChem.GetMorganFingerprintAsBitVect(mol, self.cfg.morgan_radius, nBits=self.cfg.morgan_n_bits)

    def morgan_fp_with_bit_info(self, mol: Chem.Mol):
        '''Same fingerprint as `morgan_fp`, but also returns RDKit's bitInfo dict
        (bit -> [(center_atom_idx, radius), ...]), needed to map bits back to atoms
        for the interpretability analysis in Part E.6.'''
        bit_info: Dict[int, list] = {}
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol, self.cfg.morgan_radius, nBits=self.cfg.morgan_n_bits, bitInfo=bit_info
        )
        return fp, bit_info

    def descriptors(self, mol: Chem.Mol) -> np.ndarray:
        vals = []
        for name in self.DESCRIPTOR_NAMES:
            fn = self._desc_fns.get(name)
            try:
                vals.append(fn(mol) if fn else np.nan)
            except Exception:
                vals.append(np.nan)
        return np.array(vals, dtype=float)

    def featurize_dataframe(self, df: pd.DataFrame, smiles_col: str = "smiles") -> dict:
        mols, fps, desc_rows, keep_idx = [], [], [], []
        for i, smi in enumerate(tqdm(df[smiles_col].tolist(), desc="Featurizing",
                                     unit="mol", leave=True)):
            mol = self.standardize(smi)
            if mol is None:
                continue
            mols.append(mol)
            fps.append(self.morgan_fp(mol))
            desc_rows.append(self.descriptors(mol))
            keep_idx.append(i)
        desc_arr = np.vstack(desc_rows) if desc_rows else np.empty((0, len(self.DESCRIPTOR_NAMES)))
        clean_df = df.iloc[keep_idx].reset_index(drop=True)
        n_dropped = len(df) - len(clean_df)
        if n_dropped:
            print(f"  dropped {n_dropped} molecules that failed standardization/parsing.")
        return {"df": clean_df, "mols": mols, "fps": fps, "descriptors": desc_arr,
                "descriptor_names": self.DESCRIPTOR_NAMES}


featurizer = RDKitFeaturizer(CONFIG)
pretrain_feat = featurizer.featurize_dataframe(pretrain_df, smiles_col="smiles")
train_feat = featurizer.featurize_dataframe(train_df, smiles_col="smiles")
screen_feat = featurizer.featurize_dataframe(screen_df, smiles_col="smiles")


# --- F7: deduplicate on the InChIKey of the STANDARDISED structure --------------------
# Two long-standing problems, one fix. (1) F6's shortlist of 25 rows held 5 compounds
# twice or three times - LigPrep protonation and tautomer variants of one molecule, which
# wastes shortlist slots and inflates the apparent number of hits. (2) Training compounds
# were deduplicated on the RAW SMILES string before RDKit standardisation, so salt and
# tautomer variants survived as separate rows and could straddle a fold boundary.
#
# The featurizer already standardises (Normalizer -> LargestFragmentChooser -> Uncharger),
# so variants of one compound collapse to the same standardised mol and therefore the same
# InChIKey. Deduplicating there fixes both.
try:
    from rdkit.Chem.inchi import MolToInchiKey
    _HAVE_INCHI = True
except Exception:
    _HAVE_INCHI = False
    print("!! InChI support not built into this RDKit; falling back to canonical SMILES "
          "of the standardised structure (weaker, but still post-standardisation).")


def _structure_key(mol) -> str:
    if _HAVE_INCHI:
        try:
            k = MolToInchiKey(mol)
            if k:
                return k
        except Exception:
            pass
    return Chem.MolToSmiles(mol)


def dedupe_featurized(feat: dict, label: str, keep: str = "first") -> dict:
    """Collapse duplicate structures, keeping one representative per InChIKey."""
    if not CONFIG.dedupe_on_inchikey or len(feat["mols"]) == 0:
        return feat
    keys = [_structure_key(m) for m in tqdm(feat["mols"], desc=f"{label}: InChIKeys",
                                             unit="mol", leave=False)]
    df = feat["df"].reset_index(drop=True).copy()
    df["_struct_key"] = keys
    keep_idx = (df.reset_index().groupby("_struct_key")["index"]
                .first().sort_values().values)
    n_before, n_after = len(df), len(keep_idx)
    if n_after == n_before:
        print(f"  {label}: no duplicate structures ({n_before:,} compounds)")
    else:
        print(f"  {label}: {n_before:,} -> {n_after:,} compounds "
              f"({n_before - n_after:,} duplicate structures merged)")
    return {
        "df": df.iloc[keep_idx].drop(columns=["_struct_key"]).reset_index(drop=True),
        "mols": [feat["mols"][i] for i in keep_idx],
        "fps": [feat["fps"][i] for i in keep_idx],
        "descriptors": feat["descriptors"][keep_idx],
        "descriptor_names": feat["descriptor_names"],
        "struct_keys": [keys[i] for i in keep_idx],
    }


print("\nDeduplicating on standardised-structure InChIKey:")
train_feat = dedupe_featurized(train_feat, "PSMA training set")
screen_feat = dedupe_featurized(screen_feat, "screening library")

pretrain_df = pretrain_feat["df"]
train_df = train_feat["df"]
screen_df = screen_feat["df"]

pretrain_fps, pretrain_desc_raw = pretrain_feat["fps"], pretrain_feat["descriptors"]
train_fps, train_desc_raw = train_feat["fps"], train_feat["descriptors"]
screen_fps, screen_desc_raw = screen_feat["fps"], screen_feat["descriptors"]
descriptor_names_raw = pretrain_feat["descriptor_names"]

print(f"ChEMBL pretraining set after featurization: {len(pretrain_df):,}")
print(f"BindingDB PSMA fine-tuning set after featurization: {len(train_df):,}")
print(f"Screening set after featurization: {len(screen_df):,}")
print(f"Raw descriptor pool: {len(descriptor_names_raw)} descriptors (before redundancy filtering below).")


In [ ]:
print(CONFIG.screening_library_path)

## 5b. Part B.1b — Descriptor curation (near-constant + correlation-cluster filtering)

Directly follows the AutoQSAR paper's two-stage descriptor curation, applied to the
broad ChEMBL pretraining set (the largest, most chemically diverse sample, so its
correlation structure generalizes best):

1. **Drop near-constant / near-zero-variance descriptors** — a descriptor where ≥90%
   of compounds share one value carries almost no information (e.g., "number of
   tetrazoles" is 0 for nearly everything).
2. **Cluster the survivors by |Pearson correlation|** (hierarchical, complete-linkage,
   cut at r = 0.8) and keep only the descriptor closest to each cluster's centroid.
   This directly addresses the collinearity in a ~200-descriptor pool without
   depending on any one model family to "figure it out" from redundant inputs.

The same descriptor columns (fit once, on the pretraining set) are then applied to
the fine-tuning and screening descriptor matrices, so everything downstream — the
shared `feature_scaler`, the NN trunk, the GBM/KPLS baselines — sees a consistent,
de-redundified feature space.


In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform


class AutoQSARDescriptorSelector:
    '''Near-constant filter + correlation-cluster redundancy reduction, as described in
    the AutoQSAR paper (Dixon et al., Future Med. Chem. 2016).'''

    def __init__(self, near_constant_frac: float = 0.90, corr_threshold: float = 0.8):
        self.near_constant_frac = near_constant_frac
        self.corr_threshold = corr_threshold
        self.selected_idx: Optional[np.ndarray] = None
        self.selected_names: Optional[List[str]] = None

    def fit(self, X: np.ndarray, names: Sequence[str]) -> "AutoQSARDescriptorSelector":
        X = np.nan_to_num(X)
        n = len(X)

        # Stage 1: near-constant filter.
        keep1 = []
        for j in range(X.shape[1]):
            col = X[:, j]
            _, counts = np.unique(col, return_counts=True)
            most_common_frac = counts.max() / n
            if most_common_frac < self.near_constant_frac and np.var(col) > 1e-8:
                keep1.append(j)
        if not keep1:
            keep1 = list(range(X.shape[1]))
        X1 = X[:, keep1]
        names1 = [names[j] for j in keep1]
        print(f"  Stage 1 (near-constant filter): {X.shape[1]} -> {X1.shape[1]} descriptors.")

        # Stage 2: correlation-cluster redundancy reduction.
        if X1.shape[1] > 1:
            corr = np.corrcoef(X1, rowvar=False)
            corr = np.nan_to_num(corr, nan=0.0)
            np.fill_diagonal(corr, 1.0)
            dist = 1.0 - np.abs(corr)
            dist = (dist + dist.T) / 2.0        # enforce exact symmetry (float noise)
            np.fill_diagonal(dist, 0.0)
            condensed = squareform(dist, checks=False)
            Z = linkage(condensed, method="complete")
            cluster_labels = fcluster(Z, t=1.0 - self.corr_threshold, criterion="distance")

            keep2 = []
            for cl in np.unique(cluster_labels):
                members = np.where(cluster_labels == cl)[0]
                if len(members) == 1:
                    keep2.append(members[0])
                    continue
                sub_corr = np.abs(corr[np.ix_(members, members)])
                centroid_scores = sub_corr.mean(axis=1)   # closest-to-centroid = highest mean similarity
                keep2.append(members[np.argmax(centroid_scores)])
            keep2 = sorted(keep2)
        else:
            keep2 = list(range(X1.shape[1]))

        self.selected_idx = np.array([keep1[j] for j in keep2])
        self.selected_names = [names[j] for j in self.selected_idx]
        print(f"  Stage 2 (correlation clustering, r<={self.corr_threshold}): "
              f"{X1.shape[1]} -> {len(self.selected_idx)} descriptors.")
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        return np.nan_to_num(X)[:, self.selected_idx]


print("Fitting AutoQSAR-style descriptor selector on the broad ChEMBL pretraining set...")
descriptor_selector = AutoQSARDescriptorSelector(corr_threshold=0.8).fit(pretrain_desc_raw, descriptor_names_raw)

pretrain_desc = descriptor_selector.transform(pretrain_desc_raw)
train_desc = descriptor_selector.transform(train_desc_raw)
screen_desc = descriptor_selector.transform(screen_desc_raw)
descriptor_names = descriptor_selector.selected_names

print(f"Final curated descriptor set ({len(descriptor_names)}): {descriptor_names[:12]}"
      f"{' ...' if len(descriptor_names) > 12 else ''}")


## 6. Part B.2 — Applicability Domain

Unchanged logic from the original pipeline: four complementary distance-from-training
metrics (Tanimoto dissimilarity on fingerprints, Manhattan distance in descriptor
space, Manhattan distance in a PCA "latent" proxy, and k-NN unfamiliarity), each
calibrated against a percentile of the **training set's own internal nearest-neighbor
distances**. Calibrated on the **PSMA-specific BindingDB fine-tuning set** — that's
the domain the final predictions need to be trustworthy within, not the broad ChEMBL
pretraining set.


In [ ]:
def fps_to_array(fps) -> np.ndarray:
    arr = np.zeros((len(fps), fps[0].GetNumBits()), dtype=np.uint8)
    for i, fp in enumerate(fps):
        DataStructs.ConvertToNumpyArray(fp, arr[i])
    return arr


def bulk_min_tanimoto_dissimilarity(query_fps, ref_fps,
                                     desc: str = "Tanimoto to nearest reference") -> np.ndarray:
    out = np.empty(len(query_fps))
    for i, qfp in enumerate(tqdm(query_fps, desc=desc, unit="mol", leave=False)):
        sims = DataStructs.BulkTanimotoSimilarity(qfp, ref_fps)
        out[i] = 1.0 - max(sims)
    return out


def train_internal_tanimoto_dissimilarity(train_fps) -> np.ndarray:
    out = np.empty(len(train_fps))
    for i, qfp in enumerate(tqdm(train_fps, desc="AD calibration: internal NN Tanimoto",
                                  unit="mol", leave=False)):
        sims = DataStructs.BulkTanimotoSimilarity(qfp, train_fps)
        sims[i] = -1
        out[i] = 1.0 - max(sims)
    return out


class ApplicabilityDomain:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.scaler: Optional[StandardScaler] = None
        self.pca: Optional[PCA] = None
        self.thresholds: dict = {}

    def fit(self, train_fps, train_desc: np.ndarray):
        self.scaler = StandardScaler().fit(np.nan_to_num(train_desc))
        train_desc_scaled = self.scaler.transform(np.nan_to_num(train_desc))

        n_comp = min(self.cfg.latent_pca_components, train_desc_scaled.shape[1], train_desc_scaled.shape[0] - 1)
        self.pca = PCA(n_components=max(n_comp, 1), random_state=self.cfg.random_state).fit(train_desc_scaled)
        train_latent = self.pca.transform(train_desc_scaled)

        tanimoto_internal = train_internal_tanimoto_dissimilarity(train_fps)
        self.thresholds["tanimoto"] = np.percentile(tanimoto_internal, self.cfg.ad_percentile_threshold)

        nn_desc_l1 = NearestNeighbors(n_neighbors=2, p=1).fit(train_desc_scaled)
        d_l1, _ = nn_desc_l1.kneighbors(train_desc_scaled)
        self.thresholds["descriptor_manhattan"] = np.percentile(d_l1[:, 1], self.cfg.ad_percentile_threshold)

        nn_latent_l1 = NearestNeighbors(n_neighbors=2, p=1).fit(train_latent)
        d_lat, _ = nn_latent_l1.kneighbors(train_latent)
        self.thresholds["latent_manhattan"] = np.percentile(d_lat[:, 1], self.cfg.ad_percentile_threshold)

        k = min(self.cfg.unfamiliarity_k + 1, len(train_desc_scaled))
        nn_unfam = NearestNeighbors(n_neighbors=k).fit(train_desc_scaled)
        d_unfam, _ = nn_unfam.kneighbors(train_desc_scaled)
        unfam_internal = d_unfam[:, 1:].mean(axis=1)
        self.thresholds["unfamiliarity"] = np.percentile(unfam_internal, self.cfg.ad_percentile_threshold)

        self._train_desc_scaled = train_desc_scaled
        self._train_latent = train_latent
        self._nn_desc_l1 = NearestNeighbors(n_neighbors=1, p=1).fit(train_desc_scaled)
        self._nn_latent_l1 = NearestNeighbors(n_neighbors=1, p=1).fit(train_latent)
        self._nn_unfam = NearestNeighbors(n_neighbors=min(self.cfg.unfamiliarity_k, len(train_desc_scaled))).fit(
            train_desc_scaled
        )
        return self

    def score(self, screen_fps, screen_desc: np.ndarray, train_fps) -> pd.DataFrame:
        screen_desc_scaled = self.scaler.transform(np.nan_to_num(screen_desc))
        screen_latent = self.pca.transform(screen_desc_scaled)

        tanimoto_diss = bulk_min_tanimoto_dissimilarity(
            screen_fps, train_fps,
            desc="AD: Tanimoto to nearest training compound")
        with tqdm(total=3, desc="AD: descriptor / latent / k-NN distances",
                  leave=False) as _ad_bar:
            desc_manhattan, _ = self._nn_desc_l1.kneighbors(screen_desc_scaled)
            _ad_bar.update(1)
            latent_manhattan, _ = self._nn_latent_l1.kneighbors(screen_latent)
            _ad_bar.update(1)
            unfam_d, _ = self._nn_unfam.kneighbors(screen_desc_scaled)
            _ad_bar.update(1)
        unfamiliarity = unfam_d.mean(axis=1)

        out = pd.DataFrame({
            "tanimoto_dissimilarity": tanimoto_diss,
            "descriptor_manhattan": desc_manhattan[:, 0],
            "latent_manhattan": latent_manhattan[:, 0],
            "unfamiliarity": unfamiliarity,
        })
        out["tanimoto_OOD"] = out["tanimoto_dissimilarity"] > self.thresholds["tanimoto"]
        out["descriptor_OOD"] = out["descriptor_manhattan"] > self.thresholds["descriptor_manhattan"]
        out["latent_OOD"] = out["latent_manhattan"] > self.thresholds["latent_manhattan"]
        out["unfamiliarity_OOD"] = out["unfamiliarity"] > self.thresholds["unfamiliarity"]
        out["n_flags_OOD"] = out[["tanimoto_OOD", "descriptor_OOD", "latent_OOD", "unfamiliarity_OOD"]].sum(axis=1)

        # F6: Tanimoto is a VETO, not one vote in four.
        #
        # F5 used `n_flags_OOD >= 3`, which let a compound pass while failing the
        # structural-similarity test - and for a congeneric target that is the flag with
        # teeth. The other three metrics compare bulk physicochemical properties, so a
        # molecule of roughly the right size and lipophilicity passes them while being
        # structurally unrelated to anything in training. Every one of F5's 15 top hits
        # was flagged tanimoto_OOD (dissimilarity 0.73-0.83 against a 0.457 threshold)
        # and every one of them was reported as in-domain. Meanwhile the Butina-grouped
        # evaluation showed that this is exactly the regime where the model fails.
        out["consensus_OOD"] = out["n_flags_OOD"] >= self.cfg.ad_consensus_min_flags
        if getattr(self.cfg, "ad_tanimoto_veto", False):
            out["consensus_OOD"] = out["consensus_OOD"] | out["tanimoto_OOD"]
        return out


ad = ApplicabilityDomain(CONFIG).fit(train_fps, train_desc)
ad_scores = ad.score(screen_fps, screen_desc, train_fps)
print("AD thresholds (95th percentile of the PSMA training set's own internal NN distances):")
for k, v in ad.thresholds.items():
    print(f"  {k}: {v:.4f}")

_n = len(ad_scores)
print(f"\nScreening library: {_n:,} compounds")
for flag in ("tanimoto_OOD", "descriptor_OOD", "latent_OOD", "unfamiliarity_OOD"):
    print(f"  {flag:>18s} fires on {ad_scores[flag].sum():>7,} "
          f"({100 * ad_scores[flag].mean():5.1f}%)")
_veto = getattr(CONFIG, "ad_tanimoto_veto", False)
_in_dom = int((~ad_scores["consensus_OOD"]).sum())
print(f"\n  Tanimoto veto: {'ON' if _veto else 'OFF'}"
      f"   (consensus_min_flags = {CONFIG.ad_consensus_min_flags})")
print(f"  -> in-domain: {_in_dom:,} / {_n:,} ({100 * _in_dom / _n:.2f}%)")
_would_be = int((ad_scores["n_flags_OOD"] < CONFIG.ad_consensus_min_flags).sum())
if _veto:
    print(f"     without the veto it would have been {_would_be:,} "
          f"({100 * _would_be / _n:.2f}%) - the veto removes "
          f"{_would_be - _in_dom:,} compounds that passed on bulk-property grounds")
    print("     while being structurally unlike anything in the training set.")
ad_scores.describe()


## 6b. Part B.2b — Which feature set actually works best here?

Both papers found that the best choice of input representation (fingerprints only,
descriptors only, or both together) is target-dependent rather than universal — the
KPLS paper found fingerprints alone beat descriptors alone on 9/10 targets, but the
*combination* won outright for one of them. Rather than assume `CONFIG.feature_set`
is right, a fast default-hyperparameter LightGBM is cross-validated on the PSMA
training data under all three options (scored on `pIC50`, the primary endpoint), and
whichever wins becomes the feature set used for every model built from here on.


In [ ]:
# --- Which representation works best? --------------------------------------------------
# Scored on the primary endpoint's labelled rows by ROC-AUC under stratified CV.
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier


def build_design_matrix(fps, desc: np.ndarray, scaler: StandardScaler, feature_set: str) -> np.ndarray:
    desc_scaled = scaler.transform(np.nan_to_num(desc))
    if feature_set == "descriptors_only":
        return desc_scaled
    fp_arr = fps_to_array(fps)
    if feature_set == "fp_only":
        return fp_arr
    if feature_set == "fp_plus_descriptors":
        return np.hstack([fp_arr, desc_scaled])
    raise ValueError(f"Unknown feature_set '{feature_set}'.")


_feature_set_options = ["fp_only", "descriptors_only", "fp_plus_descriptors"]
_quick_scaler = StandardScaler().fit(np.nan_to_num(train_desc))
_probe_col = f"y_{CONFIG.primary_endpoint}"
_probe_mask = train_df[_probe_col].notna().values
_y_probe = train_df.loc[_probe_mask, _probe_col].values.astype(int)
_skf = StratifiedKFold(n_splits=min(CONFIG.cv_folds, int(np.bincount(_y_probe).min())),
                       shuffle=True, random_state=CONFIG.random_state)

_fs_scores = {}
for fs in tqdm(_feature_set_options, desc="Feature-set bake-off", unit="option"):
    X_probe = build_design_matrix(train_fps, train_desc, _quick_scaler, fs)[_probe_mask]
    aucs = []
    for tr_i, te_i in tqdm(_skf.split(X_probe, _y_probe), total=_skf.get_n_splits(),
                            desc=f"  {fs}: CV folds", leave=False):
        m = LGBMClassifier(random_state=CONFIG.random_state, verbose=-1, n_estimators=150,
                            n_jobs=-1,
                            class_weight="balanced" if CONFIG.class_weight_balanced else None)
        m.fit(X_probe[tr_i], _y_probe[tr_i])
        aucs.append(roc_auc_score(_y_probe[te_i], m.predict_proba(X_probe[te_i])[:, 1]))
    _fs_scores[fs] = float(np.mean(aucs))
    print(f"  {fs:>22s}: mean CV ROC-AUC = {_fs_scores[fs]:.3f}")

_best_feature_set = max(_fs_scores, key=_fs_scores.get)
print(f"\nBest feature set on {CONFIG.primary_endpoint}: '{_best_feature_set}' "
      f"(CONFIG default was '{CONFIG.feature_set}')")
if _fs_scores["fp_only"] >= max(_fs_scores.values()) - 0.005:
    print("  (fp_only is within 0.005 of the best — the 91 curated descriptors are "
          "adding essentially nothing for this task, as in F6.)")
CONFIG.feature_set = _best_feature_set

## 7. Part B.3 — Design matrices

One shared feature scaler (`feature_scaler`, fit on the **broad ChEMBL pretraining
set**, since that's the larger and more chemically diverse sample) is reused to build
the model-input matrix for all three sets. Using the same scaler everywhere is what
lets the pretrained trunk's weights be meaningfully reused during fine-tuning — the
fine-tuning and screening inputs live in the exact same normalized feature space the
trunk was pretrained on. This is separate from `ad.scaler` above, which is
intentionally calibrated only on the PSMA set for domain-membership scoring.
`CONFIG.feature_set` here is whatever Part B.2b just selected.


In [ ]:
# --- Design matrices, masked label matrix, reference split -----------------------------
feature_scaler = StandardScaler().fit(np.nan_to_num(pretrain_desc))

X_pretrain = build_design_matrix(pretrain_fps, pretrain_desc, feature_scaler, CONFIG.feature_set)
y_pretrain = pretrain_df[CONFIG.chembl_pretrain_endpoint_name].values.astype(np.float32)

X_train_full = build_design_matrix(train_fps, train_desc, feature_scaler, CONFIG.feature_set)
X_screen = build_design_matrix(screen_fps, screen_desc, feature_scaler, CONFIG.feature_set)

# (n_compounds, n_endpoints), NaN where that compound has no label for that endpoint
Y_train_full = train_df[[f"y_{ep}" for ep in EPS]].values.astype(np.float32)
LABEL_MASK = np.isfinite(Y_train_full)


def source_weight_vector(df: pd.DataFrame, enabled: bool) -> np.ndarray:
    """Inverse-sqrt-frequency weight per source tag, mean-normalised. Exploratory; OFF by
    default, since discounting an over-represented filing is a judgement rather than a
    fact. Compare the pooled numbers both ways before adopting it."""
    if not enabled:
        return np.ones(len(df), dtype=np.float32)
    counts = df["source_tag"].map(df["source_tag"].value_counts()).values.astype(np.float64)
    w = 1.0 / np.sqrt(counts)
    return (w / w.mean()).astype(np.float32)


w_train_full = source_weight_vector(train_df, CONFIG.downweight_dominant_source)

# Reference split: stratified on the primary endpoint where it exists, random elsewhere,
# so the comparison with F6's random-split row stays meaningful.
_strat = np.where(np.isfinite(Y_train_full[:, 0]), Y_train_full[:, 0], 2).astype(int)
idx_tr, idx_te = train_test_split(
    np.arange(len(train_df)), test_size=CONFIG.test_size,
    random_state=CONFIG.random_state, stratify=_strat,
)
X_tr, X_te = X_train_full[idx_tr], X_train_full[idx_te]
Y_tr, Y_te = Y_train_full[idx_tr], Y_train_full[idx_te]
w_tr, w_te = w_train_full[idx_tr], w_train_full[idx_te]

print(f"Feature set: '{CONFIG.feature_set}' -> {X_train_full.shape[1]} input dimensions")
print(f"ChEMBL set (scaler / descriptor curation only): {X_pretrain.shape}")
print(f"PSMA multi-task — train: {X_tr.shape}, held-out reference test: {X_te.shape}\n")
for t, ep in enumerate(EPS):
    print(f"  {ep}: train {int(np.isfinite(Y_tr[:, t]).sum()):4d} labelled "
          f"({int(np.nansum(Y_tr[:, t])):3d} active) | "
          f"test {int(np.isfinite(Y_te[:, t]).sum()):4d} labelled "
          f"({int(np.nansum(Y_te[:, t])):3d} active)")
print(f"\nTotal label instances: {int(LABEL_MASK.sum()):,} across "
      f"{len(train_df):,} compounds")
if CONFIG.downweight_dominant_source:
    print(f"Source down-weighting ON: weights {w_train_full.min():.3f}-{w_train_full.max():.3f}")

## 8. Part C.1 — Multi-task classifier

A shared MLP trunk with **one binary head per endpoint**, trained with a **masked**
cross-entropy so a compound contributes through whichever heads it has labels for. That
masking is what makes the 229 pKi-only compounds usable.

The double-weighting that distorted F3–F5 — dual-labelled compounds counting twice, which
gave one patent 27% of the gradient signal from 17% of the molecules — is no longer
material, because the derived-value removal in Part A.2c leaves only ~30 compounds with two
genuinely independent labels.

Class weights are computed **per task** on that task's labelled rows, since the two
endpoints have different base rates. `use_pretraining` is False by default, so the trunk
starts from random initialisation and the frozen/unfrozen schedule is skipped entirely —
there is nothing to thaw. Part E.2d records the measurements behind that default.

In [ ]:
# --- Multi-task classifier, optional transfer --------------------------------------------
class MultiTaskClassifier(nn.Module):
    """Shared MLP trunk, one binary classification head per endpoint.

    `pretrain_head` is a single-output regression head, used only if
    CONFIG.use_pretraining is True (the ChEMBL potency labels are real numbers, so
    regression is the right pretext task). With pretraining off it is never touched and
    the trunk starts from random initialisation.
    """

    def __init__(self, n_in: int, hidden_dims=(256, 128), dropout: float = 0.2,
                 n_tasks: int = 2, n_classes: int = 2):
        super().__init__()
        layers = []
        d = n_in
        for h in hidden_dims:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.trunk = nn.Sequential(*layers)
        self.trunk_out_dim = d
        self.n_tasks = n_tasks
        self.n_classes = n_classes
        self.pretrain_head = nn.Linear(d, 1)
        self.task_heads = nn.ModuleList([nn.Linear(d, n_classes) for _ in range(n_tasks)])

    def forward_pretrain(self, x):
        return self.pretrain_head(self.trunk(x)).squeeze(-1)

    def forward_logits(self, x):
        """(batch, n_tasks, n_classes)"""
        z = self.trunk(x)
        return torch.stack([h(z) for h in self.task_heads], dim=1)


def masked_multitask_ce(logits, targets, class_weights=None, sample_weights=None):
    """Cross-entropy averaged only over (row, task) pairs that have a label.

    This is what lets a compound with only a pKi measurement contribute: it trains the pKi
    head and leaves the pIC50 head alone. After the derived-value removal in Part A.2c
    only ~30 compounds carry two labels, so the double-weighting that distorted F3-F5 (one
    patent at 27% of the gradient from 17% of the molecules) is no longer material.
    """
    n_tasks = logits.shape[1]
    total, count = 0.0, 0.0
    for t in range(n_tasks):
        tgt = targets[:, t]
        m = ~torch.isnan(tgt)
        if m.sum() == 0:
            continue
        cw = None if class_weights is None else class_weights[t]
        per = nn.functional.cross_entropy(
            logits[m, t, :], tgt[m].long(), weight=cw, reduction="none")
        if sample_weights is not None:
            sw = sample_weights[m]
            total = total + (per * sw).sum()
            count = count + sw.sum()
        else:
            total = total + per.sum()
            count = count + m.sum()
    if count == 0:
        return torch.tensor(0.0, requires_grad=True, device=logits.device)
    return total / count


def pretrain_trunk(X, y, hidden_dims, dropout, lr, weight_decay, epochs, batch_size,
                   seed=0, device=DEVICE, desc="Pretraining trunk on broad ChEMBL data",
                   leave=True) -> MultiTaskClassifier:
    torch.manual_seed(seed)
    model = MultiTaskClassifier(X.shape[1], hidden_dims=hidden_dims, dropout=dropout,
                                n_tasks=len(EPS), n_classes=N_CLASSES).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    yt = torch.tensor(y, dtype=torch.float32, device=device)
    n = len(X)
    for _ in tqdm(range(epochs), desc=desc, unit="epoch", leave=leave):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = nn.functional.mse_loss(model.forward_pretrain(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
    return model


def class_weights_per_task(Y: np.ndarray, enabled: bool):
    """Balanced class weights computed per task on that task's labelled rows only."""
    if not enabled:
        return None
    out = []
    for t in range(Y.shape[1]):
        col = Y[:, t]
        lab = col[np.isfinite(col)].astype(int)
        if len(lab) == 0:
            out.append(torch.ones(N_CLASSES, dtype=torch.float32, device=DEVICE))
            continue
        counts = np.bincount(lab, minlength=N_CLASSES).astype(np.float64)
        counts[counts == 0] = 1.0
        w = len(lab) / (N_CLASSES * counts)
        out.append(torch.tensor(w, dtype=torch.float32, device=DEVICE))
    return out


def finetune_multitask(pretrained_trunk_state, X, Y, hidden_dims, dropout, lr, weight_decay,
                       freeze_epochs, total_epochs, batch_size, sample_weight=None,
                       seed=0, device=DEVICE, desc=None, leave=False) -> MultiTaskClassifier:
    """Trains the multi-task classifier. `pretrained_trunk_state=None` means random init,
    which is the default in F7 and also the ablation's control arm. With no pretrained
    trunk there is nothing to thaw, so the freeze schedule is skipped entirely."""
    torch.manual_seed(seed)
    model = MultiTaskClassifier(X.shape[1], hidden_dims=hidden_dims, dropout=dropout,
                                n_tasks=Y.shape[1], n_classes=N_CLASSES)
    transferring = pretrained_trunk_state is not None
    if transferring:
        model.trunk.load_state_dict(pretrained_trunk_state)
    model = model.to(device)

    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    Yt = torch.tensor(Y, dtype=torch.float32, device=device)
    if sample_weight is None:
        sample_weight = np.ones(len(X), dtype=np.float32)
    swt = torch.tensor(sample_weight, dtype=torch.float32, device=device)
    cws = class_weights_per_task(Y, CONFIG.class_weight_balanced)
    n = len(X)

    if desc is None:
        desc = ("Training multi-task head (transfer)" if transferring
                else "Training multi-task classifier (random init)")
    loss = None
    _bar = tqdm(range(total_epochs), desc=desc, unit="epoch", leave=leave)
    for ep in _bar:
        frozen = transferring and ep < freeze_epochs
        for p in model.trunk.parameters():
            p.requires_grad = not frozen
        params = [p for p in model.parameters() if p.requires_grad]
        opt = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = masked_multitask_ce(model.forward_logits(Xt[idx]), Yt[idx],
                                       class_weights=cws, sample_weights=swt[idx])
            loss.backward()
            opt.step()
        _bar.set_postfix(loss=("n/a" if loss is None else f"{loss.item():.3f}"),
                         trunk="frozen" if frozen else "free", refresh=False)
    _bar.close()
    return model


def predict_proba_mt(model, X, device=DEVICE) -> np.ndarray:
    """(n, n_tasks, n_classes), dropout off."""
    model = model.to(device).eval()
    with torch.no_grad():
        logits = model.forward_logits(torch.tensor(X, dtype=torch.float32, device=device))
        return torch.softmax(logits, dim=2).cpu().numpy()


def predict_proba_mt_mc(model, X, n_passes: int, device=DEVICE):
    """(mean, std) over stochastic passes with dropout active. Secondary signal only —
    the probability itself is the uncertainty that matters, and Part E.2e is its test."""
    model = model.to(device)
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    model.train()
    with torch.no_grad():
        probs = torch.stack([
            torch.softmax(model.forward_logits(Xt), dim=2)
            for _ in tqdm(range(n_passes), desc="MC-dropout passes", unit="pass",
                           leave=False)
        ], dim=0).cpu().numpy()
    model.eval()
    return probs.mean(axis=0), probs.std(axis=0)


print(f"MultiTaskClassifier defined: {len(EPS)} heads x {N_CLASSES} classes "
      f"({', '.join(EPS)}).")
print(f"Transfer learning: {'ON' if CONFIG.use_pretraining else 'OFF — random init'}"
      f"{'' if CONFIG.use_pretraining else ' (see Part E.2d for the measurements)'}")
print(f"Training device: {DEVICE}.")

## 9. Part C.2 — Baselines: LightGBM and kernel PLS-DA, per endpoint

Two non-neural comparators, fitted **once per endpoint** on that endpoint's labelled rows —
trees and kernel methods have no masked-loss equivalent, so multi-task for them simply
means two models.

- **LightGBM** — `LGBMClassifier`, class-weighted.
- **Kernel PLS-DA** — Nystroem low-rank kernel map (An, Sherman & Dixon, *J. Chem. Inf.
  Model.* 2013), then PLS to a small latent space, then multinomial logistic regression on
  those components. Landmarks are clamped to the sample count, because leave-one-cluster-out
  produces small training halves and Nystroem cannot draw more landmarks than it has rows.

Both get stratified bootstrap ensembles so a resample cannot lose a class, and each exposes
a `*_predict_proba` wrapper that pads any class a fold never saw — so every committee member
returns the same shape and the equal-weight blend is always well defined.

In [ ]:
# --- Baselines: one LightGBM and one kernel PLS-DA per endpoint -------------------------
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression


def _labelled(Y, t):
    m = np.isfinite(Y[:, t])
    return m, Y[m, t].astype(int)


def fit_lgbm_classifier(X, y, params: dict, sample_weight=None, seed: int = 42):
    model = LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1,
                           class_weight="balanced" if CONFIG.class_weight_balanced else None,
                           **params)
    model.fit(X, y, sample_weight=sample_weight)
    return model


def fit_kpls_classifier(X, y, params: dict, sample_weight=None, seed: int = 42):
    """Nystroem kernel map -> PLS latent space -> logistic layer (PLS-DA).

    Landmarks are clamped to the sample count: leave-one-cluster-out produces small
    training halves and Nystroem cannot draw more landmarks than it has rows."""
    n_landmarks = int(min(200, max(2, len(X) - 1)))
    nys = Nystroem(kernel="rbf", gamma=params["gamma"], n_components=n_landmarks,
                   random_state=seed)
    Z = nys.fit_transform(X)
    n_comp = max(1, min(params["n_components"], min(50, Z.shape[1] - 1)))
    Y1h = np.zeros((len(y), N_CLASSES), dtype=float)
    Y1h[np.arange(len(y)), y] = 1.0
    pls = PLSRegression(n_components=n_comp)
    pls.fit(Z, Y1h)
    logit = LogisticRegression(max_iter=2000,
                               class_weight="balanced" if CONFIG.class_weight_balanced
                               else None)
    logit.fit(pls.transform(Z), y, sample_weight=sample_weight)
    return {"nystroem": nys, "pls": pls, "logit": logit, "classes_": logit.classes_}


def _align_proba(p: np.ndarray, classes) -> np.ndarray:
    """Pad to the full class set, so a fold that never saw a class still returns (n, 2)."""
    if p.shape[1] == N_CLASSES and list(classes) == list(range(N_CLASSES)):
        return p
    out = np.zeros((p.shape[0], N_CLASSES))
    for j, c in enumerate(classes):
        out[:, int(c)] = p[:, j]
    return out


def lgbm_predict_proba(model, X) -> np.ndarray:
    return _align_proba(model.predict_proba(X), model.classes_)


def kpls_predict_proba(model, X) -> np.ndarray:
    T = model["pls"].transform(model["nystroem"].transform(X))
    return _align_proba(model["logit"].predict_proba(T), model["classes_"])


def _stratified_bootstrap_idx(y, rng):
    by_class = {c: np.flatnonzero(y == c) for c in np.unique(y)}
    return np.concatenate([rng.choice(v, size=len(v), replace=True)
                           for v in by_class.values()])


def bootstrap_lgbm(X, y, params, n_bootstrap, sample_weight=None, seed=42) -> List:
    rng = np.random.RandomState(seed)
    out = []
    for b in tqdm(range(n_bootstrap), desc="Bootstrap LightGBM", unit="model", leave=False):
        i = _stratified_bootstrap_idx(y, rng)
        sw = None if sample_weight is None else sample_weight[i]
        out.append(fit_lgbm_classifier(X[i], y[i], params, sample_weight=sw, seed=seed + b))
    return out


def bootstrap_kpls(X, y, params, n_bootstrap, sample_weight=None, seed=42) -> List:
    rng = np.random.RandomState(seed)
    out = []
    for b in tqdm(range(n_bootstrap), desc="Bootstrap KPLS-DA", unit="model", leave=False):
        i = _stratified_bootstrap_idx(y, rng)
        sw = None if sample_weight is None else sample_weight[i]
        try:
            out.append(fit_kpls_classifier(X[i], y[i], params, sample_weight=sw,
                                           seed=seed + b))
        except Exception:
            continue
    return out


print(f"Baselines defined, fitted per endpoint: {EPS}")

## 9b. Part C.2b — Metrics

Fixed before any model is fitted, so the evaluation protocol is not chosen after seeing
results.

`classification_metrics` returns ROC-AUC, PR-AUC, balanced accuracy, MCC, precision,
recall, F1, Brier and ECE for one (fold, model, endpoint) row, returning `NaN` rather than
raising on a single-class fold.

The two that matter most for a screen: **`enrichment_factor`** is the hit rate in the top
fraction of a ranking divided by the overall hit rate (EF = 1 is worthless; EF = 10 at 1%
means the top percentile is ten times richer in actives), and **`precision_at_n`** is the
blunter form — of the top n ranked, what fraction are genuinely active. Note the small-fold
trap: at n = 30, `EF@1%` is computed over a single compound, so only the pooled row's
enrichment figures carry information.

`blend_equal` is the committee combiner. F5 fitted simplex weights and lost to a plain
average in all six scheme × endpoint comparisons, by up to 0.08 R² under grouped splits —
and fitted them on the very test set it then scored against.

In [ ]:
# --- Metrics, fixed before any model is fitted -----------------------------------------
from sklearn.metrics import (roc_auc_score, average_precision_score, balanced_accuracy_score,
                              matthews_corrcoef, brier_score_loss, precision_score,
                              recall_score, f1_score)


def _auc_safe(y_true, p_pos) -> float:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, p_pos))


def enrichment_factor(y_true, scores, frac: float) -> float:
    """Hit rate in the top `frac` divided by the overall hit rate. EF = 1 is a worthless
    ranking. Beware small folds: at n = 14, the top 1% is one compound."""
    y_true = np.asarray(y_true)
    base = (y_true == POSITIVE_CLASS).mean()
    if base == 0 or len(y_true) == 0:
        return float("nan")
    k = max(1, int(round(frac * len(y_true))))
    top = np.argsort(-np.asarray(scores))[:k]
    return float((y_true[top] == POSITIVE_CLASS).mean() / base)


def precision_at_n(y_true, scores, n: int) -> float:
    y_true = np.asarray(y_true)
    n = min(n, len(y_true))
    if n == 0:
        return float("nan")
    top = np.argsort(-np.asarray(scores))[:n]
    return float((y_true[top] == POSITIVE_CLASS).mean())


def expected_calibration_error(y_bin, p, n_bins=10) -> float:
    edges = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(p)
    if n == 0:
        return float("nan")
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi) if hi < 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / n) * abs(y_bin[m].mean() - p[m].mean())
    return float(ece)


def classification_metrics(y_true, p_pos, decision_threshold: float = 0.5) -> Dict[str, float]:
    """One (fold, model, endpoint) row. Returns NaN rather than raising on a single-class
    fold — leave-one-cluster-out will produce them."""
    y_true = np.asarray(y_true).astype(int)
    p_pos = np.asarray(p_pos, dtype=float)
    y_pred = (p_pos >= decision_threshold).astype(int)
    multi = len(np.unique(y_true)) > 1
    out = {
        "n": int(len(y_true)),
        "n_active": int((y_true == POSITIVE_CLASS).sum()),
        "base_rate": float((y_true == POSITIVE_CLASS).mean()) if len(y_true) else float("nan"),
        "roc_auc": _auc_safe(y_true, p_pos),
        "pr_auc": float(average_precision_score(y_true, p_pos)) if multi else float("nan"),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)) if multi else float("nan"),
        "mcc": float(matthews_corrcoef(y_true, y_pred)) if multi else float("nan"),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier": float(brier_score_loss(y_true, np.clip(p_pos, 0, 1))) if len(y_true) else float("nan"),
        "ECE": expected_calibration_error(y_true, np.clip(p_pos, 0, 1)) if len(y_true) else float("nan"),
    }
    for frac in CONFIG.enrichment_fractions:
        out[f"EF@{int(frac * 100)}%"] = enrichment_factor(y_true, p_pos, frac)
    return out


def blend_equal(p_list: List[np.ndarray]) -> np.ndarray:
    """Equal-weight average of P(active) vectors. F5's fitted simplex weights lost to a
    plain average in all six scheme x endpoint comparisons, by up to 0.08 R2 under grouped
    splits, and were fitted on the very test set they were scored against."""
    return np.mean(np.stack([np.asarray(p, dtype=float) for p in p_list], axis=0), axis=0)


print("Metrics defined: classification_metrics, enrichment_factor, precision_at_n,")
print("expected_calibration_error, blend_equal.")
print(f"EF at {', '.join(f'{int(f * 100)}%' for f in CONFIG.enrichment_fractions)}; "
      f"precision@{list(CONFIG.precision_at_n)}.")

## 10. Part D.1 — Hyperparameter tuning: LightGBM and KPLS-DA, per endpoint

Grid search against Bayesian search for LightGBM, a small hand-rolled grid for KPLS-DA,
stratified folds, ROC-AUC, ranked by the AutoQSAR gap-aware score — run separately for each
endpoint on that endpoint's labelled rows.

The gap penalty systematically prefers under-fit models: on F4's pKi data it chose a
50-tree, 7-leaf LightGBM over a configuration with a better held-out score purely because
its train/test gap was narrower. Both candidates and both raw AUCs are printed, and the cell
says so explicitly when the AutoQSAR winner is not the model with the better held-out AUC.

In [ ]:
# --- Hyperparameter tuning, per endpoint ------------------------------------------------
import numpy as np
from sklearn.model_selection import GridSearchCV
from skopt import BayesSearchCV
from skopt.space import Real, Integer


def autoqsar_score(acc_train: float, acc_test: float) -> float:
    """AutoQSAR's gap-aware score (Dixon et al. 2016), on ROC-AUC with a 0.5 floor.

    Caveat carried from the F4 review: this penalty systematically prefers under-fit
    models. Both candidates and both raw AUCs are printed, and the cell says so when the
    AutoQSAR winner is not the model with the better held-out score."""
    if acc_test < 0.5:
        return acc_test - 0.5
    return (acc_test - 0.5) * (1.0 - abs(acc_train - acc_test)) + 0.5


def pick_best_by_autoqsar_score(cv_results: dict):
    tr, te = cv_results["mean_train_score"], cv_results["mean_test_score"]
    aq = [autoqsar_score(a, b) for a, b in zip(tr, te)]
    i = int(np.argmax(aq))
    return i, aq[i], float(tr[i]), float(te[i])


GRID_PARAMS = {"n_estimators": [100, 200, 400], "learning_rate": [0.01, 0.05, 0.1],
               "num_leaves": [15, 31, 63], "max_depth": [-1, 6, 10]}
BAYES_SPACE = {"n_estimators": Integer(50, 500),
               "learning_rate": Real(0.005, 0.3, prior="log-uniform"),
               "num_leaves": Integer(7, 63), "max_depth": Integer(3, 12),
               "min_child_samples": Integer(5, 40)}
KPLS_GRID = {"n_components": [2, 4, 6, 8, 12],
             "gamma": [1.0 / X_tr.shape[1], 0.01, 0.05, 0.1]}

gbm_best_params, kpls_best_params = {}, {}
gbm_tuning_results, kpls_tuning_results = {}, {}

for t, ep in enumerate(tqdm(EPS, desc="Hyperparameter tuning", unit="endpoint")):
    m, y_ep = _labelled(Y_tr, t)
    Xs = X_tr[m]
    n_splits = min(CONFIG.cv_folds, int(np.bincount(y_ep).min()))
    if n_splits < 2:
        tqdm.write(f"\n=== {ep}: too few of one class to tune; using defaults ===")
        gbm_best_params[ep] = {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 31}
        kpls_best_params[ep] = {"n_components": 4, "gamma": 1.0 / X_tr.shape[1]}
        continue
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=CONFIG.random_state)
    tqdm.write(f"\n=== {ep}: {len(y_ep)} labelled training rows "
               f"({int(y_ep.sum())} active), {n_splits}-fold stratified CV ===")

    cands = {}
    bar = tqdm(total=int(CONFIG.run_gridsearch) + int(CONFIG.run_bayes_search) + 1,
               desc=f"  {ep}: searches", unit="search", leave=False)
    if CONFIG.run_gridsearch:
        gcv = GridSearchCV(
            LGBMClassifier(random_state=CONFIG.random_state, verbose=-1, n_jobs=-1,
                           class_weight="balanced" if CONFIG.class_weight_balanced else None),
            GRID_PARAMS, cv=skf, scoring="roc_auc", n_jobs=-1, return_train_score=True)
        gcv.fit(Xs, y_ep)
        i, aq, a, b = pick_best_by_autoqsar_score(gcv.cv_results_)
        cands["GridSearchCV"] = (aq, gcv.cv_results_["params"][i], a, b)
        bar.update(1)
        tqdm.write(f"  GridSearchCV  AutoQSAR={aq:.3f} (train {a:.3f}, test {b:.3f})  "
                   f"{gcv.cv_results_['params'][i]}")
    if CONFIG.run_bayes_search:
        bcv = BayesSearchCV(
            LGBMClassifier(random_state=CONFIG.random_state, verbose=-1, n_jobs=-1,
                           class_weight="balanced" if CONFIG.class_weight_balanced else None),
            BAYES_SPACE, n_iter=CONFIG.bayes_search_n_iter, cv=skf, scoring="roc_auc",
            n_jobs=-1, random_state=CONFIG.random_state, return_train_score=True)
        bcv.fit(Xs, y_ep)
        i, aq, a, b = pick_best_by_autoqsar_score(bcv.cv_results_)
        cands["BayesSearchCV"] = (aq, dict(bcv.cv_results_["params"][i]), a, b)
        bar.update(1)
        tqdm.write(f"  BayesSearchCV AutoQSAR={aq:.3f} (train {a:.3f}, test {b:.3f})  "
                   f"{dict(bcv.cv_results_['params'][i])}")

    win = max(cands, key=lambda k: cands[k][0])
    gbm_best_params[ep] = cands[win][1]
    gbm_tuning_results[ep] = cands
    tqdm.write(f"  -> LightGBM winner: {win} (AutoQSAR={cands[win][0]:.3f})")
    by_auc = max(cands, key=lambda k: cands[k][3])
    if by_auc != win:
        tqdm.write(f"     NOTE: {by_auc} had the better test AUC "
                   f"({cands[by_auc][3]:.3f} vs {cands[win][3]:.3f}); the gap penalty "
                   f"chose the narrower-gap model.")

    rows = []
    for nc in KPLS_GRID["n_components"]:
        for gm in KPLS_GRID["gamma"]:
            tr_a, te_a = [], []
            for a_i, b_i in skf.split(Xs, y_ep):
                try:
                    k = fit_kpls_classifier(Xs[a_i], y_ep[a_i],
                                            {"n_components": nc, "gamma": gm},
                                            seed=CONFIG.random_state)
                    tr_a.append(_auc_safe(y_ep[a_i], kpls_predict_proba(k, Xs[a_i])[:, 1]))
                    te_a.append(_auc_safe(y_ep[b_i], kpls_predict_proba(k, Xs[b_i])[:, 1]))
                except Exception:
                    continue
            if not te_a:
                continue
            a, b = float(np.nanmean(tr_a)), float(np.nanmean(te_a))
            rows.append({"n_components": nc, "gamma": gm, "train_auc": a, "test_auc": b,
                         "autoqsar": autoqsar_score(a, b)})
    bar.update(1)
    bar.close()
    res = pd.DataFrame(rows).sort_values("autoqsar", ascending=False)
    kpls_tuning_results[ep] = res
    if len(res):
        r0 = res.iloc[0]
        kpls_best_params[ep] = {"n_components": int(r0["n_components"]),
                                "gamma": float(r0["gamma"])}
        tqdm.write(f"  KPLS-DA grid  AutoQSAR={r0['autoqsar']:.3f} "
                   f"(train {r0['train_auc']:.3f}, test {r0['test_auc']:.3f})  "
                   f"{kpls_best_params[ep]}")
    else:
        kpls_best_params[ep] = {"n_components": 4, "gamma": 1.0 / X_tr.shape[1]}
        tqdm.write(f"  KPLS-DA grid failed; default {kpls_best_params[ep]}")

print("\nFinal tuned params:")
for ep in EPS:
    print(f"  {ep}: LightGBM {gbm_best_params[ep]}")
    print(f"  {ep}: KPLS-DA  {kpls_best_params[ep]}")

## 11. Part D.2 — Hyperparameter tuning: the network (Optuna, TPE)

Tree-structured Parzen Estimator over architecture and training hyperparameters, scored by
**mean held-out ROC-AUC across both endpoints** on a stratified validation slice of the
training split.

With `use_pretraining = False` there is no pretraining pass and `freeze_epochs` is fixed at
0 — it only exists to stage the thawing of a transferred trunk, and there isn't one.

Watch the dropout. F5's search chose 0.047, which left MC-dropout with no variance to report
and was part of why its uncertainty came out four times too tight; F6's chose 0.364 and the
spread became meaningful. The cell warns below 0.1 — not a bug to override, but a reason to
trust the recalibrated probability and Part E.2c rather than the MC standard deviation.

In [ ]:
# --- Optuna TPE search for the network --------------------------------------------------
# Objective: mean held-out ROC-AUC across endpoints on a validation slice of the training
# split. With pretraining off, `freeze_epochs` has nothing to thaw and is fixed at 0.
_strat_opt = np.where(np.isfinite(Y_tr[:, 0]), Y_tr[:, 0], 2).astype(int)
_i_tr, _i_val = train_test_split(
    np.arange(len(Y_tr)), test_size=1.0 / CONFIG.optuna_cv_folds,
    random_state=CONFIG.random_state, stratify=_strat_opt)
X_opt_tr, X_opt_val = X_tr[_i_tr], X_tr[_i_val]
Y_opt_tr, Y_opt_val = Y_tr[_i_tr], Y_tr[_i_val]
w_opt_tr = w_tr[_i_tr]

_default_hidden = tuple(CONFIG.nn_hidden_dims)
_quick_pretrain_epochs = max(5, CONFIG.nn_pretrain_epochs // 4)
_base_pretrained = None
if CONFIG.use_pretraining:
    _base_pretrained = pretrain_trunk(
        X_pretrain, y_pretrain, hidden_dims=_default_hidden, dropout=CONFIG.nn_dropout,
        lr=CONFIG.nn_lr, weight_decay=CONFIG.nn_weight_decay,
        epochs=_quick_pretrain_epochs, batch_size=CONFIG.nn_pretrain_batch_size,
        seed=CONFIG.random_state, desc="Optuna: base pretrain pass")
else:
    print("use_pretraining = False -> no pretraining pass; trunks start from random init.")


def _mean_val_auc(model) -> float:
    P = predict_proba_mt(model, X_opt_val)
    scores = []
    for t in range(Y_opt_val.shape[1]):
        m = np.isfinite(Y_opt_val[:, t])
        if m.sum() > 3:
            a = _auc_safe(Y_opt_val[m, t].astype(int), P[m, t, POSITIVE_CLASS])
            if np.isfinite(a):
                scores.append(a)
    return float(np.mean(scores)) if scores else 0.5


def optuna_objective(trial: "optuna.Trial") -> float:
    h1 = trial.suggest_categorical("h1", [64, 128, 256])
    h2 = trial.suggest_categorical("h2", [32, 64, 128])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    freeze_epochs = (trial.suggest_int("freeze_epochs", 0, 30)
                     if CONFIG.use_pretraining else 0)

    hidden_dims = (h1, h2)
    trunk_state = None
    if CONFIG.use_pretraining:
        trunk_state = (_base_pretrained.trunk.state_dict() if hidden_dims == _default_hidden
                       else pretrain_trunk(X_pretrain, y_pretrain, hidden_dims=hidden_dims,
                                           dropout=dropout, lr=CONFIG.nn_lr,
                                           weight_decay=weight_decay,
                                           epochs=_quick_pretrain_epochs,
                                           batch_size=CONFIG.nn_pretrain_batch_size,
                                           seed=CONFIG.random_state,
                                           desc=f"Optuna trial {trial.number}: pretrain",
                                           leave=False).trunk.state_dict())

    model = finetune_multitask(
        trunk_state, X_opt_tr, Y_opt_tr, hidden_dims=hidden_dims, dropout=dropout,
        lr=lr, weight_decay=weight_decay, freeze_epochs=freeze_epochs,
        total_epochs=max(30, min(CONFIG.nn_finetune_total_epochs, 60)),
        batch_size=batch_size, sample_weight=w_opt_tr, seed=CONFIG.random_state,
        desc=f"Optuna trial {trial.number}", leave=False)
    return _mean_val_auc(model)


if CONFIG.run_optuna:
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=CONFIG.random_state))
    study.optimize(optuna_objective, n_trials=CONFIG.optuna_n_trials, show_progress_bar=True)
    nn_best_params = dict(study.best_params)
    nn_best_params.setdefault("freeze_epochs", 0)
    print(f"Optuna best mean validation ROC-AUC = {study.best_value:.3f}")
    print(f"Optuna best params: {nn_best_params}")
    if nn_best_params.get("dropout", 1.0) < 0.1:
        print("\n!! dropout below 0.1 — MC-dropout will report almost no spread (the F5")
        print("   failure). Trust the recalibrated probability and Part E.2e, not the MC std.")
else:
    nn_best_params = {"h1": CONFIG.nn_hidden_dims[0], "h2": CONFIG.nn_hidden_dims[1],
                      "dropout": CONFIG.nn_dropout, "lr": CONFIG.nn_finetune_lr,
                      "weight_decay": CONFIG.nn_weight_decay, "freeze_epochs": 0,
                      "batch_size": CONFIG.nn_batch_size}
    print("Optuna disabled — CONFIG defaults:", nn_best_params)

nn_best_params

## 12. Part E.1 — Train the final models

The network is trained twice — once on the stratified random training split for the
reference numbers at the end of this section, once on the full set for the models that score
the library — and each baseline family is fitted per endpoint on both, plus stratified
bootstrap ensembles.

The reference scores are **not** the headline. A stratified random split of a set made of a
handful of congeneric families tests mostly on close analogues of the training compounds;
that is the flaw F4 quantified and Part E.2 replaces. In F6 the gap was ROC-AUC 0.89 random
against 0.69 pooled leave-one-cluster-out.

In [ ]:
# --- Train the final models ---------------------------------------------------------------
nn_hidden_dims_final = (nn_best_params["h1"], nn_best_params["h2"])

_PRETRAINED_STATE = None
pretrained_final = None
if CONFIG.use_pretraining:
    pretrained_final = pretrain_trunk(
        X_pretrain, y_pretrain, hidden_dims=nn_hidden_dims_final,
        dropout=nn_best_params["dropout"], lr=CONFIG.nn_lr,
        weight_decay=nn_best_params["weight_decay"], epochs=CONFIG.nn_pretrain_epochs,
        batch_size=CONFIG.nn_pretrain_batch_size, seed=CONFIG.random_state,
        desc="Final trunk pretraining (full ChEMBL set)")
    _PRETRAINED_STATE = pretrained_final.trunk.state_dict()


def _train_nn(X, Y, w, trunk_state, seed, desc, leave=False):
    return finetune_multitask(
        trunk_state, X, Y, hidden_dims=nn_hidden_dims_final,
        dropout=nn_best_params["dropout"], lr=nn_best_params["lr"],
        weight_decay=nn_best_params["weight_decay"],
        freeze_epochs=nn_best_params.get("freeze_epochs", 0),
        total_epochs=CONFIG.nn_finetune_total_epochs,
        batch_size=nn_best_params["batch_size"], sample_weight=w,
        seed=seed, desc=desc, leave=leave)


nn_model_eval = _train_nn(X_tr, Y_tr, w_tr, _PRETRAINED_STATE, CONFIG.random_state,
                          "Reference model (random split)", leave=True)
nn_model_deploy = _train_nn(X_train_full, Y_train_full, w_train_full, _PRETRAINED_STATE,
                            CONFIG.random_state, "Deploy model (full PSMA set)", leave=True)

gbm_eval, gbm_deploy, gbm_boot = {}, {}, {}
kpls_eval, kpls_deploy, kpls_boot = {}, {}, {}
for t, ep in enumerate(tqdm(EPS, desc="Per-endpoint baselines", unit="endpoint")):
    m_tr, y_ep_tr = _labelled(Y_tr, t)
    m_all, y_ep_all = _labelled(Y_train_full, t)
    gp, kp = gbm_best_params[ep], kpls_best_params[ep]

    gbm_eval[ep] = fit_lgbm_classifier(X_tr[m_tr], y_ep_tr, gp,
                                       sample_weight=w_tr[m_tr], seed=CONFIG.random_state)
    gbm_deploy[ep] = fit_lgbm_classifier(X_train_full[m_all], y_ep_all, gp,
                                         sample_weight=w_train_full[m_all],
                                         seed=CONFIG.random_state)
    gbm_boot[ep] = bootstrap_lgbm(X_train_full[m_all], y_ep_all, gp,
                                  n_bootstrap=CONFIG.n_bootstrap,
                                  sample_weight=w_train_full[m_all], seed=CONFIG.random_state)
    try:
        kpls_eval[ep] = fit_kpls_classifier(X_tr[m_tr], y_ep_tr, kp,
                                           sample_weight=w_tr[m_tr], seed=CONFIG.random_state)
        kpls_deploy[ep] = fit_kpls_classifier(X_train_full[m_all], y_ep_all, kp,
                                             sample_weight=w_train_full[m_all],
                                             seed=CONFIG.random_state)
        kpls_boot[ep] = bootstrap_kpls(X_train_full[m_all], y_ep_all, kp,
                                       n_bootstrap=CONFIG.n_bootstrap,
                                       sample_weight=w_train_full[m_all],
                                       seed=CONFIG.random_state)
    except Exception as e:
        print(f"  {ep}: KPLS-DA failed ({type(e).__name__}); excluded")
        kpls_eval[ep] = kpls_deploy[ep] = None
        kpls_boot[ep] = []

print(f"\nFinal models trained. Bootstrap ensembles per endpoint: "
      + ", ".join(f"{ep} {len(gbm_boot[ep])} GBM / {len(kpls_boot[ep])} KPLS" for ep in EPS))

# --- reference-split scores, comparable with F6's random-split row ----------------------
_P_nn = predict_proba_mt(nn_model_eval, X_te)
ref_rows = []
for t, ep in enumerate(EPS):
    m = np.isfinite(Y_te[:, t])
    if m.sum() < 5 or len(np.unique(Y_te[m, t])) < 2:
        continue
    y_true = Y_te[m, t].astype(int)
    preds = {"NN": _P_nn[m, t, POSITIVE_CLASS],
             "LightGBM": lgbm_predict_proba(gbm_eval[ep], X_te[m])[:, POSITIVE_CLASS]}
    if kpls_eval.get(ep) is not None:
        preds["KPLS-DA"] = kpls_predict_proba(kpls_eval[ep], X_te[m])[:, POSITIVE_CLASS]
    preds["Blend (all)"] = blend_equal(list(preds.values()))
    if "KPLS-DA" in preds:
        preds["Blend (GBM+KPLS)"] = blend_equal([preds["LightGBM"], preds["KPLS-DA"]])
    for name, p in preds.items():
        ref_rows.append({"endpoint": ep, "model": name, **classification_metrics(y_true, p)})

ref_eval_df = pd.DataFrame(ref_rows).set_index(["endpoint", "model"]).round(3)
print("\nSingle stratified random split — REFERENCE ONLY. Part E.2's leave-one-cluster-out")
print("is the evaluation that reflects prospective use:\n")
display(ref_eval_df)

## 13. Part E.2 — Leave-one-cluster-out evaluation

Whole chemical families are held out one at a time, so the result reads as "generalises to
these families, fails on that one" rather than as an average that describes neither.

**What changed from F6.** F6 reported eight folds of which four could not be scored:
cluster 6 had zero actives (every metric `NaN`), cluster 4 had one active in 18 compounds
(ROC-AUC 0.059 — a single compound landing low in a ranking, not a measurement), cluster 1
had 14 compounds at a 71% base rate (AUC 1.000, nearly free at that balance). Averaging
those produced a macro row that meant very little, and it also made the
distance-versus-difficulty correlation (r = −0.056) look like a finding when it was really
an underpowered test.

A cluster now earns its own row only if it clears **both** `loco_min_cluster_size` and
`loco_min_fold_actives` on at least one endpoint. Everything smaller is distributed across
`loco_n_remainder_folds` size-balanced "small families" folds. Every compound still
receives an out-of-fold prediction, so pooled coverage stays complete — only the per-fold
table changes, and it becomes readable.

**Three aggregates.** **Pooled** concatenates all out-of-fold predictions and scores them
once; it weights every compound equally and is the number to quote. **Macro** averages the
per-fold scores and is shown only because the spread across families is itself
informative. **Random split** runs alongside purely so the leakage gap to F5/F6
methodology stays visible — in F6 that gap was 0.89 → 0.69 ROC-AUC.

**Five committee members.** The network, LightGBM, KPLS-DA, the three-way equal blend, and
a two-way LightGBM + KPLS-DA blend. The last is new: in F6 KPLS-DA alone beat the three-way
average on the honest evaluation (AUC 0.706 vs 0.690, precision@50 0.58 vs 0.52), which
raises the question of whether the network is diluting the other two. Everything is scored
**per endpoint**, since a compound may have a label for one and not the other.

In [ ]:
# --- Build the folds -------------------------------------------------------------------
from rdkit.ML.Cluster import Butina
from sklearn.model_selection import StratifiedShuffleSplit


def butina_cluster_groups(fps, cutoff: float = 0.4) -> np.ndarray:
    n = len(fps)
    dists: List[float] = []
    for i in tqdm(range(1, n), desc=f"Butina distances (cutoff={cutoff})", leave=False):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend(1.0 - s for s in sims)
    clusters = Butina.ClusterData(dists, n, cutoff, isDistData=True)
    groups = np.empty(n, dtype=int)
    for ci, members in enumerate(clusters):
        for idx in members:
            groups[idx] = ci
    return groups


butina_groups = butina_cluster_groups(train_fps, cutoff=CONFIG.butina_cutoff)
_sizes = pd.Series(butina_groups).value_counts()
print(f"Butina clustering of {len(train_df):,} compounds (cutoff {CONFIG.butina_cutoff}):")
print(f"  {len(_sizes)} clusters | largest {_sizes.iloc[0]} "
      f"({100 * _sizes.iloc[0] / len(butina_groups):.1f}%) | "
      f"median {int(_sizes.median())} | singletons {int((_sizes == 1).sum())}")


def _fold_is_scorable(mask) -> bool:
    """At least `loco_min_fold_actives` of BOTH classes on at least one endpoint."""
    for t in range(Y_train_full.shape[1]):
        col = Y_train_full[mask, t]
        lab = col[np.isfinite(col)]
        if (lab == 1).sum() >= CONFIG.loco_min_fold_actives and \
           (lab == 0).sum() >= CONFIG.loco_min_fold_actives:
            return True
    return False


own_fold, pooled_clusters, rejected = [], [], []
for c in _sizes.index:
    c = int(c)
    mask = butina_groups == c
    if _sizes[c] < CONFIG.loco_min_cluster_size:
        pooled_clusters.append(c)
    elif not _fold_is_scorable(mask):
        pooled_clusters.append(c)
        rejected.append((c, int(_sizes[c])))
    elif len(own_fold) < CONFIG.loco_max_clusters:
        own_fold.append(c)
    else:
        pooled_clusters.append(c)

print(f"\n  {len(own_fold)} clusters get their own fold "
      f"(>= {CONFIG.loco_min_cluster_size} compounds and "
      f">= {CONFIG.loco_min_fold_actives} of each class)")
if rejected:
    print(f"  {len(rejected)} large-enough clusters pooled for being too unbalanced "
          f"to score: " + ", ".join(f"cluster {c} (n={n})" for c, n in rejected[:6]))
print(f"  {len(pooled_clusters)} clusters pooled into "
      f"{CONFIG.loco_n_remainder_folds} size-balanced 'small families' folds")

# greedy size-balanced assignment of the pooled clusters
_bins = [[] for _ in range(max(1, CONFIG.loco_n_remainder_folds))]
_bin_sizes = [0] * len(_bins)
for c in sorted(pooled_clusters, key=lambda c: -_sizes[c]):
    j = int(np.argmin(_bin_sizes))
    _bins[j].append(c)
    _bin_sizes[j] += int(_sizes[c])

LOCO_FOLDS = []
for c in own_fold:
    LOCO_FOLDS.append({"fold": f"cluster {c}", "mask": butina_groups == c})
for j, cs in enumerate(_bins):
    if not cs:
        continue
    LOCO_FOLDS.append({"fold": f"small families #{j + 1}",
                       "mask": np.isin(butina_groups, cs)})

# metadata
for f in tqdm(LOCO_FOLDS, desc="Fold metadata", unit="fold"):
    m = f["mask"]
    sub = train_df[m]
    f["n"] = int(m.sum())
    for t, ep in enumerate(EPS):
        col = Y_train_full[m, t]
        lab = col[np.isfinite(col)]
        f[f"{ep}_n"] = int(len(lab))
        f[f"{ep}_active"] = int((lab == 1).sum())
    srcs = sub["source_tag"].value_counts()
    f["dominant_source"] = srcs.index[0] if len(srcs) else "?"
    f["source_share"] = round(float(srcs.iloc[0] / len(sub)), 2) if len(srcs) else np.nan
    d = bulk_min_tanimoto_dissimilarity(
        [train_fps[i] for i in np.flatnonzero(m)],
        [train_fps[i] for i in np.flatnonzero(~m)],
        desc=f"  {f['fold']}: distance to train")
    f["mean_tanimoto_to_train"] = round(float(np.mean(d)), 3)

loco_folds_df = pd.DataFrame([{k: v for k, v in f.items() if k != "mask"}
                              for f in LOCO_FOLDS]).set_index("fold")
fold_of = np.full(len(train_df), -1, dtype=int)
for i, f in enumerate(LOCO_FOLDS):
    fold_of[f["mask"]] = i

print(f"\n{len(LOCO_FOLDS)} folds covering "
      f"{int((fold_of >= 0).sum()):,} of {len(train_df):,} compounds:")
display(loco_folds_df)

_single = loco_folds_df[(loco_folds_df["source_share"] >= 0.6)
                        & (~loco_folds_df["dominant_source"].isin(GENERIC_SOURCE_TAGS))]
if len(_single):
    print("Folds dominated by a single filing — read these carefully, because holding one")
    print("out removes a large, chemically narrow block of training data at the same time")
    print("as it poses an unfamiliar test set:")
    for name, r in _single.iterrows():
        print(f"  {name}: {r['n']} compounds, {100 * r['source_share']:.0f}% from "
              f"{r['dominant_source']}, mean Tanimoto distance {r['mean_tanimoto_to_train']}")

In [ ]:
# --- Run the folds ----------------------------------------------------------------------
MODEL_ORDER = ["NN", "LightGBM", "KPLS-DA", "Blend (all)", "Blend (GBM+KPLS)"]
if CONFIG.run_pretraining_ablation:
    MODEL_ORDER = ["NN", "NN (transfer)"] + MODEL_ORDER[1:]

_ABLATION_STATE = None
if CONFIG.run_pretraining_ablation:
    if _PRETRAINED_STATE is not None:
        _ABLATION_STATE = _PRETRAINED_STATE
    else:
        print("run_pretraining_ablation = True but use_pretraining = False: pretraining a "
              "trunk purely for the comparison.")
        _abl = pretrain_trunk(X_pretrain, y_pretrain, hidden_dims=nn_hidden_dims_final,
                              dropout=nn_best_params["dropout"], lr=CONFIG.nn_lr,
                              weight_decay=nn_best_params["weight_decay"],
                              epochs=CONFIG.nn_pretrain_epochs,
                              batch_size=CONFIG.nn_pretrain_batch_size,
                              seed=CONFIG.random_state,
                              desc="Ablation: pretraining a trunk for comparison")
        _ABLATION_STATE = _abl.trunk.state_dict()


def _fit_committee(tr_i, te_i, seed, label):
    """Returns {model: {endpoint: p_active on te_i}}. Members that cannot be fitted for an
    endpoint are omitted for that endpoint rather than for the whole fold."""
    Xa, Ya, wa = X_train_full[tr_i], Y_train_full[tr_i], w_train_full[tr_i]
    Xb = X_train_full[te_i]
    out: Dict[str, Dict[str, np.ndarray]] = {m: {} for m in MODEL_ORDER}

    nn_arms = [("NN", _PRETRAINED_STATE if CONFIG.use_pretraining else None)]
    if CONFIG.run_pretraining_ablation:
        nn_arms.append(("NN (transfer)", _ABLATION_STATE))
    for name, trunk in nn_arms:
        mdl = _train_nn(Xa, Ya, wa, trunk, seed, f"  {label}: {name}", leave=False)
        P = predict_proba_mt(mdl, Xb)
        for t, ep in enumerate(EPS):
            out[name][ep] = P[:, t, POSITIVE_CLASS]

    for t, ep in enumerate(EPS):
        m_a = np.isfinite(Ya[:, t])
        y_a = Ya[m_a, t].astype(int)
        if m_a.sum() < 10 or len(np.unique(y_a)) < 2:
            continue
        g = fit_lgbm_classifier(Xa[m_a], y_a, gbm_best_params[ep],
                                sample_weight=wa[m_a], seed=seed)
        out["LightGBM"][ep] = lgbm_predict_proba(g, Xb)[:, POSITIVE_CLASS]
        try:
            k = fit_kpls_classifier(Xa[m_a], y_a, kpls_best_params[ep],
                                    sample_weight=wa[m_a], seed=seed)
            out["KPLS-DA"][ep] = kpls_predict_proba(k, Xb)[:, POSITIVE_CLASS]
        except Exception as e:
            tqdm.write(f"  {label}/{ep}: KPLS-DA failed ({type(e).__name__})")

        members = [out["NN"][ep]] if ep in out["NN"] else []
        if ep in out["LightGBM"]:
            members.append(out["LightGBM"][ep])
        if ep in out["KPLS-DA"]:
            members.append(out["KPLS-DA"][ep])
        if len(members) >= 2:
            out["Blend (all)"][ep] = blend_equal(members)
        pair = [out[m][ep] for m in ("LightGBM", "KPLS-DA") if ep in out[m]]
        if len(pair) == 2:
            out["Blend (GBM+KPLS)"][ep] = blend_equal(pair)
    return out


# out-of-fold P(active), one vector per (model, endpoint)
oof = {m: {ep: np.full(len(train_df), np.nan) for ep in EPS} for m in MODEL_ORDER}
loco_rows = []

for fi, f in enumerate(tqdm(LOCO_FOLDS, desc="Leave-one-cluster-out", unit="fold")):
    te_i = np.flatnonzero(f["mask"])
    tr_i = np.flatnonzero(~f["mask"])
    if not np.isfinite(Y_train_full[tr_i]).any():
        tqdm.write(f"  {f['fold']}: no labels in the training half; skipped")
        continue
    probas = _fit_committee(tr_i, te_i, CONFIG.random_state + 7000 + fi, f["fold"])
    for name in MODEL_ORDER:
        for t, ep in enumerate(EPS):
            p = probas[name].get(ep)
            if p is None:
                continue
            oof[name][ep][te_i] = p
            m_lab = np.isfinite(Y_train_full[te_i, t])
            if m_lab.sum() < 5:
                continue
            loco_rows.append({
                "fold": f["fold"], "endpoint": ep, "model": name,
                "dominant_source": f["dominant_source"],
                "mean_tanimoto_to_train": f["mean_tanimoto_to_train"],
                **classification_metrics(Y_train_full[te_i][m_lab, t].astype(int),
                                         p[m_lab]),
            })

loco_eval_df = pd.DataFrame(loco_rows)
print(f"\nCollected {len(loco_eval_df)} (fold, endpoint, model) results.")

# --- random-split reference -------------------------------------------------------------
rand_rows = []
_strat_ref = np.where(np.isfinite(Y_train_full[:, 0]), Y_train_full[:, 0], 2).astype(int)
_sss = StratifiedShuffleSplit(n_splits=CONFIG.loco_random_reference_repeats,
                              test_size=CONFIG.test_size, random_state=CONFIG.random_state)
for r, (tr_i, te_i) in enumerate(tqdm(list(_sss.split(X_train_full, _strat_ref)),
                                       desc="Random-split reference", unit="split")):
    probas = _fit_committee(tr_i, te_i, CONFIG.random_state + 9000 + r, f"random {r}")
    for name in MODEL_ORDER:
        for t, ep in enumerate(EPS):
            p = probas[name].get(ep)
            if p is None:
                continue
            m_lab = np.isfinite(Y_train_full[te_i, t])
            if m_lab.sum() < 5:
                continue
            rand_rows.append({
                "repeat": r, "endpoint": ep, "model": name,
                **classification_metrics(Y_train_full[te_i][m_lab, t].astype(int),
                                         p[m_lab]),
            })

random_eval_df = pd.DataFrame(rand_rows)
print(f"Collected {len(random_eval_df)} (repeat, endpoint, model) random-split results.")

## 13b. Part E.2b — Results, per family and pooled

Read the **pooled** block first — it is the cross-validated estimate, with every compound
scored once on a prediction made while its whole chemical family was absent from training.
The **per-fold** block says *which* families the model reaches and which it does not.
**Macro** is shown only to expose the spread.

For a screening application, read `precision@N` and `EF` before `roc_auc`. "Of the top 50
ranked, this fraction were genuinely active" is the number that decides whether a shortlist
earns assay time; AUC can look respectable while the top of the ranking is useless.

One caution on the small folds: `EF@1%` on a fold of 30 compounds is computed over a single
compound and should be ignored. The pooled row is the only place the enrichment figures are
meaningful.

In [ ]:
# --- Summaries ---------------------------------------------------------------------------
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

_metrics = [c for c in ["roc_auc", "pr_auc", "balanced_acc", "mcc", "precision", "recall",
                        "brier", "ECE"] if c in loco_eval_df.columns]
_efs = [c for c in loco_eval_df.columns if c.startswith("EF@")]
_models = [m for m in MODEL_ORDER if m in set(loco_eval_df["model"])]

print("=" * 110)
print("PER-FOLD RESULTS")
print("=" * 110)
display(loco_eval_df.set_index(["endpoint", "fold", "model"])[
    ["n", "n_active", "base_rate", "mean_tanimoto_to_train"] + _metrics + _efs].round(3))

print("\n" + "=" * 110)
print("POOLED — out-of-fold predictions concatenated and scored once. QUOTE THESE.")
print("=" * 110)
pooled_rows = []
for ep_i, ep in enumerate(EPS):
    y_col = Y_train_full[:, ep_i]
    for name in _models:
        p = oof[name][ep]
        ok = np.isfinite(p) & np.isfinite(y_col)
        if ok.sum() < 20 or len(np.unique(y_col[ok])) < 2:
            continue
        met = classification_metrics(y_col[ok].astype(int), p[ok])
        met.update({"endpoint": ep, "model": name, "coverage": int(ok.sum())})
        met.update({f"precision@{k}": precision_at_n(y_col[ok].astype(int), p[ok], k)
                    for k in CONFIG.precision_at_n})
        pooled_rows.append(met)
pooled_df = pd.DataFrame(pooled_rows).set_index(["endpoint", "model"]).round(3)
display(pooled_df)

print("\n" + "=" * 110)
print("MACRO — mean +/- s.d. across folds (spread, not an estimate)")
print("=" * 110)
macro_df = (loco_eval_df.groupby(["endpoint", "model"])[_metrics + _efs]
            .agg(["mean", "std"]).round(3))
display(macro_df)

print("\n" + "=" * 110)
print("RANDOM-SPLIT REFERENCE — the gap to pooled is the leakage")
print("=" * 110)
rand_summary = (random_eval_df.groupby(["endpoint", "model"])[_metrics + _efs]
                .agg(["mean", "std"]).round(3))
display(rand_summary)

print("\nHeadline — ROC-AUC, random split vs leave-one-cluster-out (pooled):")
for ep in EPS:
    print(f"\n  {ep}:")
    for name in _models:
        try:
            pl = pooled_df.loc[(ep, name), "roc_auc"]
            rd = rand_summary.loc[(ep, name), ("roc_auc", "mean")]
        except KeyError:
            continue
        print(f"    {name:<18s} random {rd:.3f}  ->  pooled {pl:.3f}   [{pl - rd:+.3f}]")
print("\nThe drop is the share of the random-split score that came from testing on close")
print("analogues. The pooled column is what bears on unfamiliar chemistry.")

# does the committee beat its members, and is the network diluting it?
print("\n" + "-" * 110)
for ep in EPS:
    try:
        sub = pooled_df.loc[ep, "roc_auc"].sort_values(ascending=False)
    except KeyError:
        continue
    print(f"{ep}: best pooled ROC-AUC -> {sub.index[0]} ({sub.iloc[0]:.3f})")
    if "Blend (GBM+KPLS)" in sub.index and "Blend (all)" in sub.index:
        d = sub["Blend (GBM+KPLS)"] - sub["Blend (all)"]
        if d > 0.01:
            print(f"   the 2-model blend beats the 3-model blend by {d:+.3f} — the network "
                  f"is diluting the committee on this endpoint")
        elif d < -0.01:
            print(f"   the network is contributing: 3-model blend beats the pair by {-d:+.3f}")
        else:
            print(f"   2-model and 3-model blends are within 0.01 — the network adds nothing "
                  f"measurable but costs nothing either")

## 13c. Part E.2c — Recalibrating the probabilities

F6's probabilities were roughly twice too high: mean predicted 0.50 against an observed
rate of 0.265, with expected calibration error of 0.24–0.31 across models. The cause is
`class_weight="balanced"`, which by construction pushes outputs toward 0.5 in order to
weight the minority class up. The *ranking* was fine — that is what ROC-AUC measures — but
the numbers were scores, not probabilities, and quoting one as "a 53% chance of being
active" would have been wrong.

The fix is a monotone post-hoc map from score to probability, fitted **cross-fitted across
the folds**: for each fold, the calibrator is trained on the out-of-fold predictions of
every *other* fold and then applied to this one. That matters — fitting a calibrator on the
same predictions you then evaluate makes the result look better than it is, which is the
same mistake F5 made with its blend weights.

**Platt scaling is the default**, not isotonic. The miscalibration being corrected is a
systematic ~2× inflation caused by the balanced class weighting, which is close to a pure
logit shift — precisely what a two-parameter sigmoid models. Isotonic regression is more
flexible, but it fits a step function, and with only a handful of folds it overfits and
collapses distinct scores into ties. In principle a monotone map cannot change a ranking or
an AUC; in practice isotonic's ties can, and a smoke test on noise features moved AUC by
0.10 that way. The cell reports the largest AUC change so this is visible rather than
silent — anything beyond rounding means ties were collapsed. Set
`CONFIG.recalibration = "isotonic"` if the before/after table shows a miscalibration shape
a sigmoid cannot follow.

A final calibrator fitted on all out-of-fold predictions is then applied to the library
scores in Part E.4, so the P(active) column there means what it says.

In [ ]:
# --- Cross-fitted probability recalibration ----------------------------------------------
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression as _LR


class _Calibrator:
    """Monotone score -> probability map. Cannot change a ranking, only the numbers."""

    def __init__(self, method="isotonic"):
        self.method = method
        self._m = None

    def fit(self, p, y):
        p, y = np.asarray(p, dtype=float), np.asarray(y, dtype=int)
        ok = np.isfinite(p) & np.isfinite(y)
        p, y = p[ok], y[ok]
        if len(p) < 20 or len(np.unique(y)) < 2 or self.method == "none":
            self._m = None
            return self
        if self.method == "isotonic":
            self._m = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0).fit(p, y)
        else:
            self._m = _LR(max_iter=1000).fit(p.reshape(-1, 1), y)
        return self

    def transform(self, p):
        p = np.asarray(p, dtype=float)
        if self._m is None:
            return p
        out = np.full_like(p, np.nan)
        ok = np.isfinite(p)
        if self.method == "isotonic":
            out[ok] = self._m.predict(p[ok])
        else:
            out[ok] = self._m.predict_proba(p[ok].reshape(-1, 1))[:, 1]
        return np.clip(out, 0.0, 1.0)

    @property
    def slope(self) -> float:
        """Sign of the score -> probability relationship the calibrator learned.

        A NEGATIVE slope means the calibrator inverted the ranking: on the folds it was
        fitted to, a higher model score went with a LOWER chance of being active. That is
        not a calibration detail, it is a finding about the model - it was anti-correlated
        with the label on those folds. Surfaced below rather than silently applied."""
        if self._m is None:
            return float("nan")
        if self.method == "isotonic":
            y = np.asarray(self._m.y_thresholds_).ravel()
            return float(y[-1] - y[0]) if len(y) > 1 else 0.0
        return float(np.ravel(self._m.coef_)[0])


METHOD = CONFIG.recalibration
print(f"Recalibration method: '{METHOD}' (cross-fitted across "
      f"{len(LOCO_FOLDS)} leave-one-cluster-out folds)\n")

oof_cal = {m: {ep: np.full(len(train_df), np.nan) for ep in EPS} for m in MODEL_ORDER}
final_calibrators: Dict[tuple, _Calibrator] = {}
fold_slopes: List[dict] = []

for ep_i, ep in enumerate(EPS):
    y_col = Y_train_full[:, ep_i]
    for name in MODEL_ORDER:
        p = oof[name][ep]
        if not np.isfinite(p).any():
            continue
        # cross-fitted: fold f calibrated on every other fold
        for fi in range(len(LOCO_FOLDS)):
            in_f = fold_of == fi
            out_f = (fold_of != fi) & (fold_of >= 0)
            tr_ok = out_f & np.isfinite(p) & np.isfinite(y_col)
            if tr_ok.sum() < 20:
                oof_cal[name][ep][in_f] = p[in_f]
                continue
            cal = _Calibrator(METHOD).fit(p[tr_ok], y_col[tr_ok].astype(int))
            oof_cal[name][ep][in_f] = cal.transform(p[in_f])
            fold_slopes.append({"model": name, "endpoint": ep, "fold": fi,
                                "slope": cal.slope, "n_fit": int(tr_ok.sum())})
        # deployment calibrator: all out-of-fold predictions
        all_ok = np.isfinite(p) & np.isfinite(y_col)
        final_calibrators[(name, ep)] = _Calibrator(METHOD).fit(
            p[all_ok], y_col[all_ok].astype(int))

# --- before / after ---------------------------------------------------------------------
cal_rows = []
for ep_i, ep in enumerate(EPS):
    y_col = Y_train_full[:, ep_i]
    for name in MODEL_ORDER:
        raw, cal = oof[name][ep], oof_cal[name][ep]
        ok = np.isfinite(raw) & np.isfinite(cal) & np.isfinite(y_col)
        if ok.sum() < 20 or len(np.unique(y_col[ok])) < 2:
            continue
        yb = y_col[ok].astype(int)
        cal_rows.append({
            "endpoint": ep, "model": name, "n": int(ok.sum()),
            "observed_rate": round(float(yb.mean()), 3),
            "mean_pred_raw": round(float(raw[ok].mean()), 3),
            "mean_pred_cal": round(float(cal[ok].mean()), 3),
            "ECE_raw": round(expected_calibration_error(yb, np.clip(raw[ok], 0, 1)), 4),
            "ECE_cal": round(expected_calibration_error(yb, np.clip(cal[ok], 0, 1)), 4),
            "brier_raw": round(float(brier_score_loss(yb, np.clip(raw[ok], 0, 1))), 4),
            "brier_cal": round(float(brier_score_loss(yb, np.clip(cal[ok], 0, 1))), 4),
            "auc_raw": round(_auc_safe(yb, raw[ok]), 4),
            "auc_cal": round(_auc_safe(yb, cal[ok]), 4),
        })

recalibration_df = pd.DataFrame(cal_rows).set_index(["endpoint", "model"])
display(recalibration_df)

_imp = recalibration_df["ECE_raw"] - recalibration_df["ECE_cal"]
print(f"ECE improved for {int((_imp > 0).sum())} of {len(_imp)} (model, endpoint) pairs; "
      f"mean reduction {_imp.mean():+.4f}")
_auc_shift = (recalibration_df["auc_cal"] - recalibration_df["auc_raw"]).abs().max()
print(f"Largest absolute AUC change from calibration: {_auc_shift:.4f}")
if _auc_shift > 0.01:
    if METHOD == "isotonic":
        print("   A monotone map cannot reorder, so a shift this large means isotonic")
        print("   collapsed distinct scores into ties — it is overfitting the folds. Try")
        print("   CONFIG.recalibration = 'sigmoid'.")
    else:
        print("   A sigmoid is STRICTLY monotone and cannot reorder, so a shift this large")
        print("   means some calibrator fitted a NEGATIVE slope and inverted the ranking.")
        print("   See the inversion check below — that is a finding about the model, not")
        print("   about the calibration.")

# --- did any calibrator invert the ranking? ---------------------------------------------
# Checked on the CROSS-FITTED per-fold calibrators, which are what actually produced the
# calibrated out-of-fold column, not only on the deployment calibrators.
slopes_df = pd.DataFrame(fold_slopes)
_inv_fold = slopes_df[slopes_df["slope"] < 0] if len(slopes_df) else pd.DataFrame()
_inv_final = [(n, e, c.slope) for (n, e), c in final_calibrators.items()
              if np.isfinite(c.slope) and c.slope < 0]

if len(_inv_fold) or _inv_final:
    print("\n!! RANKING INVERSION — a calibrator learned that a higher model score goes "
          "with")
    print("   a LOWER chance of being active: the model was anti-correlated with the label")
    print("   on the rows used to fit it. That is a finding about the model, not about the")
    print("   calibration.")
    if len(_inv_fold):
        summ = (_inv_fold.groupby(["endpoint", "model"])
                .agg(inverted_folds=("fold", "count"), worst_slope=("slope", "min"))
                .round(3))
        _tot = slopes_df.groupby(["endpoint", "model"])["fold"].count()
        summ["total_folds"] = [int(_tot.get((e, m), 0)) for e, m in summ.index]
        print("\n   cross-fitted (per-fold) calibrators:")
        display(summ)
        print("   These are what produced the calibrated out-of-fold column, so the AUC")
        print("   shift reported above comes from here.")
    if _inv_final:
        print("\n   deployment calibrators (applied to the library in Part E.3):")
        for n, e, sl in sorted(_inv_final, key=lambda r: r[2]):
            print(f"     {n:<18s} / {e:<6s}  slope {sl:+.3f}")
        print("   !! A negative deployment slope means the library ranking is INVERTED for")
        print("      that model. Do not ship it — check the pooled AUC for that model first.")
    print("\n   Expect inversions only where the pooled AUC is at or below ~0.5: there the")
    print("   model has no signal and the calibrator is fitting noise. Cross-reference the")
    print("   pooled table before reading anything into it.")
else:
    print("   No calibrator inverted its ranking (all slopes positive, per-fold and "
          "deployment).")

if (recalibration_df["ECE_cal"] > 0.1).any():
    print("\n!! Some pairs still exceed ECE 0.1 after calibration.")
    if METHOD == "sigmoid":
        print("   A 2-parameter sigmoid cannot follow an arbitrary miscalibration shape;")
        print("   try CONFIG.recalibration = 'isotonic' and compare — but check the AUC")
        print("   shift above, since isotonic can overfit folds this small.")
    else:
        print("   Try CONFIG.recalibration = 'sigmoid' (Platt): two parameters instead of a")
        print("   step function, and strictly monotone so it cannot collapse ties.")

# --- reliability diagram, before and after ----------------------------------------------
fig, axes = plt.subplots(1, len(EPS) + 1, figsize=(6.2 * (len(EPS) + 1), 5.0))
axes = np.atleast_1d(axes)
_cols = plt.get_cmap("tab10")

for ep_i, ep in enumerate(EPS):
    ax = axes[ep_i]
    ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="perfect")
    y_col = Y_train_full[:, ep_i]
    for style, store, lbl in [(":", oof, "raw"), ("-", oof_cal, "calibrated")]:
        for mi, name in enumerate(["Blend (all)", "LightGBM", "KPLS-DA"]):
            if name not in store:
                continue
            p = store[name][ep]
            ok = np.isfinite(p) & np.isfinite(y_col)
            if ok.sum() < 40:
                continue
            yb = (y_col[ok] == POSITIVE_CLASS).astype(int)
            pv = np.clip(p[ok], 0, 1)
            try:
                bins = pd.qcut(pv, q=min(8, max(2, ok.sum() // 25)), duplicates="drop")
            except Exception:
                continue
            xs, ys = [], []
            for b in bins.categories:
                sel = bins == b
                if sel.sum() < 5:
                    continue
                xs.append(pv[sel].mean()); ys.append(yb[sel].mean())
            if xs:
                ax.plot(xs, ys, style, marker="o", markersize=4, color=_cols(mi),
                        linewidth=1.5, alpha=0.9 if lbl == "calibrated" else 0.5,
                        label=f"{name} ({lbl})")
    ax.set_xlabel("mean predicted P(active)")
    ax.set_ylabel("observed fraction active")
    ax.set_title(f"{ep}: reliability, out-of-fold\n(dotted = raw, solid = calibrated)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.25)
    ax.legend(fontsize=7)

ax = axes[-1]
ep0 = EPS[0]
p = oof_cal["Blend (all)"][ep0]
y_col = Y_train_full[:, 0]
ok = np.isfinite(p) & np.isfinite(y_col)
for c, nm in enumerate(CLASS_NAMES):
    sel = ok & (y_col == c)
    if sel.sum():
        ax.hist(np.clip(p[sel], 0, 1), bins=25, alpha=0.6, label=f"{nm} (n={int(sel.sum())})")
ax.set_xlabel(f"calibrated P(active) — blend, {ep0}")
ax.set_ylabel("compounds")
ax.set_title("Separation between the true classes")
ax.legend(fontsize=8); ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 13d. Part E.2d — Transfer learning: the record

`use_pretraining` is **off** in F7. The measurements behind that, from two runs on the same
data under different objectives:

| run | objective | scheme | ΔR²/ΔAUC (transfer − scratch) | repeats favouring | p |
|---|---|---|---|---|---|
| F4 | regression | random | +0.062 ± 0.025 | 10 / 10 | 0.002 |
| F4 | regression | scaffold | +0.025 ± 0.170 | 4 / 10 | 0.92 |
| F4 | regression | Butina | +0.165 ± 0.323 | 6 / 10 | 0.38 |
| F6 | classification | random | −0.0066 ± 0.0090 | 1 / 10 | 0.020 |
| F6 | classification | Butina LOCO | −0.0095 ± 0.0615 | 4 / 7 | 1.00 |

A small, perfectly consistent *benefit* when interpolating within known families under
regression; a small, consistent *harm* under classification; and nothing detectable under
either objective once whole families are held out — which is the regime that matters for a
screen. Effect sizes are under 0.01 AUC and the sign depends on the objective, so the
honest summary is that pretraining on 6,547 related-target ChEMBL bioactivities does
nothing measurable for this task.

Dropping it removes the pretraining pass, the frozen/unfrozen schedule and `freeze_epochs`
from the search space. The ChEMBL set is still used for descriptor curation and for fitting
the feature scaler, where its diversity genuinely helps and no PSMA labels are involved, so
there is no leakage.

Set `run_pretraining_ablation = True` to re-measure it on the current data: the cell below
then reports the paired comparison on these folds rather than citing the table above.

In [ ]:
# --- Paired transfer-vs-scratch ablation, if it was run ---------------------------------
from scipy import stats

if not CONFIG.run_pretraining_ablation:
    print("run_pretraining_ablation = False — nothing measured on this run.")
    print("The F4/F6 measurements in the markdown above are the basis for "
          "use_pretraining = False.")
    print("Set CONFIG.run_pretraining_ablation = True to re-measure on this data "
          "(costs one pretraining pass plus one extra network per fold).")
    ablation_df = pd.DataFrame()
else:
    rows = []
    for scheme, df, key in [("leave-one-cluster-out", loco_eval_df, "fold"),
                            ("random split", random_eval_df, "repeat")]:
        for ep in EPS:
            sub = df[df["endpoint"] == ep]
            piv = sub.pivot_table(index=key, columns="model", values="roc_auc")
            if "NN" not in piv.columns or "NN (transfer)" not in piv.columns:
                continue
            d = (piv["NN (transfer)"] - piv["NN"]).dropna()
            if len(d) < 3:
                continue
            try:
                p = float(stats.wilcoxon(d, zero_method="zsplit").pvalue)
            except Exception:
                p = float("nan")
            rows.append({"scheme": scheme, "endpoint": ep, "n_pairs": int(len(d)),
                         "mean_dAUC": round(float(d.mean()), 4),
                         "std_dAUC": round(float(d.std()), 4),
                         "favouring_transfer": int((d > 0).sum()),
                         "wilcoxon_p": round(p, 4) if np.isfinite(p) else np.nan})
    ablation_df = pd.DataFrame(rows).set_index(["scheme", "endpoint"])
    display(ablation_df)
    for (scheme, ep), r in ablation_df.iterrows():
        verdict = ("supports transfer learning" if r["mean_dAUC"] > abs(r["std_dAUC"])
                   else "no detectable benefit from pretraining")
        print(f"  {scheme:<22s} / {ep:<6s}: dAUC = {r['mean_dAUC']:+.3f} "
              f"+/- {r['std_dAUC']:.3f} ({int(r['favouring_transfer'])}/"
              f"{int(r['n_pairs'])}, p={r['wilcoxon_p']})  ->  {verdict}")
    print("\nPooled out-of-fold ROC-AUC by arm:")
    for ep_i, ep in enumerate(EPS):
        for name in ("NN", "NN (transfer)"):
            p = oof[name][ep]
            y_col = Y_train_full[:, ep_i]
            ok = np.isfinite(p) & np.isfinite(y_col)
            if ok.sum() > 20:
                print(f"  {ep:<6s} {name:<16s}: "
                      f"{_auc_safe(y_col[ok].astype(int), p[ok]):.3f} (n={int(ok.sum())})")

## 14. Part E.3 — Screen the library

Each committee member scores every library compound on both endpoints; the blend is their
equal-weight average; and the **deployment calibrators from Part E.2c are applied**, so the
`p_active_*` columns are probabilities rather than raw scores.

Because the derived Ki values were removed, pIC50 ≥ 8 and pKi ≥ 8 now mean the same
potency, which makes a **combined** score coherent: the mean of the two calibrated
blends. Two partly-independent endpoints agreeing is a stronger signal than either alone,
and this is the concrete payoff of restoring the multi-task formulation.

The MC-dropout and bootstrap spreads are reported as `*_spread`. They measure model
disagreement, not predictive error — the calibrated probability is the uncertainty that
matters, and Part E.2c is its test.

In [ ]:
# --- Score the library -------------------------------------------------------------------
P_nn_mean, P_nn_std = predict_proba_mt_mc(nn_model_deploy, X_screen,
                                          CONFIG.nn_mc_dropout_passes)

screen_results = screen_df.copy().reset_index(drop=True)
_combined, _combined_members = [], []

for t, ep in enumerate(tqdm(EPS, desc="Library predictions", unit="endpoint")):
    raw = {"NN": P_nn_mean[:, t, POSITIVE_CLASS]}
    spreads = [P_nn_std[:, t, POSITIVE_CLASS] ** 2]

    if gbm_boot.get(ep):
        stack = np.stack([lgbm_predict_proba(m, X_screen)[:, POSITIVE_CLASS]
                          for m in tqdm(gbm_boot[ep], desc=f"  {ep}: GBM bootstrap",
                                         unit="model", leave=False)], axis=0)
        raw["LightGBM"] = stack.mean(axis=0)
        spreads.append(stack.std(axis=0) ** 2)
    elif gbm_deploy.get(ep) is not None:
        raw["LightGBM"] = lgbm_predict_proba(gbm_deploy[ep], X_screen)[:, POSITIVE_CLASS]

    if kpls_boot.get(ep):
        stack = np.stack([kpls_predict_proba(m, X_screen)[:, POSITIVE_CLASS]
                          for m in tqdm(kpls_boot[ep], desc=f"  {ep}: KPLS-DA bootstrap",
                                         unit="model", leave=False)], axis=0)
        raw["KPLS-DA"] = stack.mean(axis=0)
        spreads.append(stack.std(axis=0) ** 2)
    elif kpls_deploy.get(ep) is not None:
        raw["KPLS-DA"] = kpls_predict_proba(kpls_deploy[ep], X_screen)[:, POSITIVE_CLASS]

    raw_blend = blend_equal(list(raw.values()))

    # apply the deployment calibrator fitted on out-of-fold predictions in Part E.2c
    cal = final_calibrators.get(("Blend (all)", ep))
    cal_blend = cal.transform(raw_blend) if cal is not None else raw_blend

    for name, p in raw.items():
        c = final_calibrators.get((name, ep))
        screen_results[f"p_{ep}_{name.replace(' ', '').replace('-', '')}"] = (
            c.transform(p) if c is not None else p)
    screen_results[f"p_active_{ep}"] = cal_blend
    screen_results[f"p_active_{ep}_raw"] = raw_blend
    screen_results[f"p_active_{ep}_spread"] = np.sqrt(np.sum(spreads, axis=0) / len(spreads))
    _combined.append(cal_blend)
    _combined_members.append(ep)

# both thresholds are 10 nM, so averaging them is meaningful
screen_results["p_active_combined"] = np.mean(np.stack(_combined, axis=0), axis=0)
screen_results["endpoints_agree"] = (
    np.min(np.stack(_combined, axis=0), axis=0) >= 0.5) if len(_combined) > 1 else np.nan

screen_results = pd.concat([screen_results, ad_scores.reset_index(drop=True)], axis=1)

print(f"Scored {len(screen_results):,} library compounds on {len(EPS)} endpoints "
      f"(calibrated with the Part E.2c maps).\n")
for ep in EPS:
    c = screen_results[f"p_active_{ep}"]
    r = screen_results[f"p_active_{ep}_raw"]
    print(f"  {ep}: calibrated median {c.median():.3f}, 99.9th pct "
          f"{c.quantile(0.999):.3f}, max {c.max():.3f}   "
          f"(raw max was {r.max():.3f})")
cc = screen_results["p_active_combined"]
print(f"  combined: median {cc.median():.3f}, max {cc.max():.3f}")
print()
for thr in (0.5, 0.7, 0.9):
    n = int((cc >= thr).sum())
    n_in = int(((cc >= thr) & ~screen_results["consensus_OOD"]).sum())
    print(f"  combined P(active) >= {thr:.1f}: {n:>7,} compounds ({n_in:,} in-domain)")
screen_results.head()

## 15. Part E.4 — Confidence against domain distance

Where do the confident predictions sit relative to the applicability domain? A compound
with high P(active) and low Tanimoto distance is a candidate. One with high P(active) and
high distance is the model extrapolating with conviction — and Part E.2's per-fold results
say how much that conviction is worth on chemistry it has no analogue for.

In F6 the model was *more* confident on unfamiliar compounds than familiar ones (mean 0.194
vs 0.139), which is the clearest possible sign that confidence was not tracking
reliability. The same check runs here on the recalibrated probabilities.

In [ ]:
# --- P(active) against domain distance ----------------------------------------------------
_SCORE = "p_active_combined"
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

ax = axes[0]
_s = screen_results.sample(min(40000, len(screen_results)), random_state=CONFIG.random_state)
sc = ax.scatter(_s["tanimoto_dissimilarity"], _s[_SCORE], c=_s["n_flags_OOD"],
                cmap="viridis", alpha=0.35, s=8, linewidths=0)
ax.axvline(ad.thresholds["tanimoto"], color="#d1495b", linestyle="--", linewidth=1.8,
           label=f"Tanimoto threshold ({ad.thresholds['tanimoto']:.3f})")
ax.set_xlabel("Tanimoto dissimilarity to nearest training compound")
ax.set_ylabel("calibrated combined P(active)")
ax.set_title("Confidence vs domain distance\n(right of the line = structurally unfamiliar)")
plt.colorbar(sc, ax=ax, label="# AD flags")
ax.legend(fontsize=8, loc="upper left")
ax.grid(alpha=0.2)

ax = axes[1]
_bins = pd.qcut(screen_results["tanimoto_dissimilarity"], q=10, duplicates="drop")
_g = screen_results.groupby(_bins, observed=True)[_SCORE]
_mid = [iv.mid for iv in _g.mean().index]
ax.plot(_mid, _g.mean().values, "o-", linewidth=1.8, label="mean P(active)")
ax.fill_between(_mid, _g.quantile(0.1).values, _g.quantile(0.9).values, alpha=0.2,
                label="10th-90th pct")
ax.axvline(ad.thresholds["tanimoto"], color="#d1495b", linestyle="--", linewidth=1.8)
ax.set_xlabel("Tanimoto dissimilarity (decile midpoint)")
ax.set_ylabel("calibrated combined P(active)")
ax.set_title("Does confidence fall further from home?")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

_far = screen_results[screen_results["tanimoto_OOD"]]
_near = screen_results[~screen_results["tanimoto_OOD"]]
print(f"Structurally familiar (tanimoto_OOD False): {len(_near):,} compounds, "
      f"mean P(active) {_near[_SCORE].mean():.3f}")
print(f"Structurally unfamiliar (tanimoto_OOD True): {len(_far):,} compounds, "
      f"mean P(active) {_far[_SCORE].mean():.3f}")
if len(_far) and len(_near):
    if _far[_SCORE].mean() >= _near[_SCORE].mean():
        print("\n!! Still no less confident on unfamiliar chemistry than on familiar "
              "chemistry.")
        print("   Confidence is not tracking reliability, so the applicability-domain "
              "flags —")
        print("   not the probability — are what must gate the shortlist.")
    else:
        print("\n   Confidence does fall on unfamiliar chemistry, which is the behaviour "
              "wanted.")
        print("   It still does not replace the AD flags, but it no longer contradicts "
              "them.")

## 16. Part E.5 — Shortlist

Ranked by combined calibrated P(active) among in-domain compounds, with the Tanimoto veto
applied. Duplicate structures were merged on standardised InChIKey in Part B.1, so each row
is a distinct compound rather than a protonation variant.

The expected yield printed underneath comes from Part E.2's pooled out-of-fold
precision@N — measured on compounds whose entire chemical family was held out. It is the
closest available estimate of what a shortlist is worth, and it is the number to give
anyone deciding whether to run an assay.

Read it alongside the applicability-domain count. If almost nothing in the library is in
domain, the shortlist is not a shortlist of hits; it is a shortlist of the least
extrapolated guesses available, and the honest conclusion is about the library rather than
about the compounds.

In [ ]:
# --- Shortlist ---------------------------------------------------------------------------
_id_cols = [c for c in ("compound_id", "smiles") if c in screen_results.columns]
_p_cols = ["p_active_combined"] + [f"p_active_{ep}" for ep in EPS] \
          + [f"p_active_{ep}_spread" for ep in EPS]
_ad_cols = ["n_flags_OOD", "consensus_OOD", "tanimoto_dissimilarity"]
cols = _id_cols + _p_cols + _ad_cols

in_domain = screen_results[~screen_results["consensus_OOD"]]
print(f"{len(in_domain):,} / {len(screen_results):,} compounds in the applicability domain "
      f"({100 * len(in_domain) / len(screen_results):.3f}%)")
print(f"  Tanimoto veto: {'ON' if CONFIG.ad_tanimoto_veto else 'OFF'}")
if len(in_domain):
    print(f"  of those, {int((in_domain['p_active_combined'] >= 0.5).sum()):,} have "
          f"combined P(active) >= 0.5")

top_hits = in_domain.sort_values("p_active_combined", ascending=False)[cols].head(25)
display(top_hits.round(4))

print("\n" + "=" * 96)
print("EXPECTED YIELD — from the pooled leave-one-cluster-out results")
print("=" * 96)
for ep in EPS:
    key = ("Blend (all)" if (ep, "Blend (all)") in pooled_df.index else None)
    if key is None:
        continue
    r = pooled_df.loc[(ep, key)]
    print(f"\n  {ep} (blend, whole families held out):")
    print(f"    ROC-AUC {r['roc_auc']:.3f} | PR-AUC {r['pr_auc']:.3f} "
          f"(base rate {r['base_rate']:.3f}) | ECE {r.get('ECE', float('nan')):.3f}")
    for n in CONFIG.precision_at_n:
        k = f"precision@{n}"
        if k in r and np.isfinite(r[k]):
            print(f"    precision@{n:<4d}: {r[k]:.3f}  -> ~{r[k] * 100:.0f}% of the top {n} "
                  f"were genuinely active")
    for c in [c for c in pooled_df.columns if c.startswith("EF@")]:
        if np.isfinite(r[c]):
            print(f"    {c:<8s}: {r[c]:.2f}x enrichment")

if len(top_hits):
    _md = top_hits["tanimoto_dissimilarity"].median()
    print(f"\n  median Tanimoto distance of this shortlist : {_md:.3f}")
    print(f"  AD Tanimoto threshold                      : {ad.thresholds['tanimoto']:.3f}")
    if _md > ad.thresholds["tanimoto"]:
        print("  !! Above the threshold even after the veto — treat the ranking as a weak")
        print("     prior, not a prediction.")

_n_in = len(in_domain)
if _n_in < 0.01 * len(screen_results):
    print(f"\n  !! Only {_n_in:,} of {len(screen_results):,} library compounds are within")
    print("     structural reach of the training set. The binding constraint on this")
    print("     exercise is the library, not the model: no modelling change can produce a")
    print("     credible shortlist from compounds the model has no basis to judge. Replacing")
    print("     the library is worth more than anything else in this notebook.")

## 16b. Part E.6 — Which atoms drive P(active)?

Per-atom attribution by **substructure ablation**: each "on" fingerprint bit is set to 0,
the calibrated blended P(active) recomputed, and the change spread over the atoms in that
bit's environment.

This corrects F5, which nudged each on-bit from 1 to 1.05 and divided by 0.05. That does
nothing to a tree ensemble — a 0.05 change in a binary feature essentially never crosses a
split threshold — so LightGBM contributed exactly zero and the map was an NN gradient
labelled as a blend. Setting the bit to 0 asks the question that means something, and every
committee member responds to it.

In [ ]:
# --- Which atoms drive P(active)? ---------------------------------------------------------
from rdkit.Chem import Draw
from collections import defaultdict


def get_bit_atoms(mol, bit_info: dict, bit_id: int) -> set:
    atoms = set()
    for atom_idx, radius in bit_info[bit_id]:
        if radius == 0:
            atoms.add(atom_idx)
        else:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
            amap: dict = {}
            Chem.PathToSubmol(mol, env, atomMap=amap)
            atoms.update(amap.keys())
    return atoms


def predict_blend_single(x_row: np.ndarray, ep_idx: int, ep: str) -> float:
    """Calibrated blended P(active) for one feature vector on one endpoint."""
    xb = x_row.reshape(1, -1).astype(np.float32)
    members = [predict_proba_mt(nn_model_deploy, xb)[0, ep_idx, POSITIVE_CLASS]]
    if gbm_deploy.get(ep) is not None:
        members.append(lgbm_predict_proba(gbm_deploy[ep], xb)[0, POSITIVE_CLASS])
    if kpls_deploy.get(ep) is not None:
        try:
            members.append(kpls_predict_proba(kpls_deploy[ep], xb)[0, POSITIVE_CLASS])
        except Exception:
            pass
    raw = float(np.mean(members))
    cal = final_calibrators.get(("Blend (all)", ep))
    return float(cal.transform(np.array([raw]))[0]) if cal is not None else raw


def compute_atom_sensitivities(mol, x_row, ep_idx, ep) -> Dict[int, float]:
    """Per-atom contribution by ABLATING each 'on' fingerprint bit (1 -> 0).

    F5 perturbed each on-bit from 1 to 1.05, which does nothing to a tree ensemble: a 0.05
    nudge on a binary feature essentially never crosses a split threshold, so LightGBM
    returned an unchanged prediction and contributed exactly zero. The map was an NN
    gradient wearing a blend's label. Setting the bit to 0 asks the meaningful question —
    "what if this substructure were absent" — and every committee member responds to it.
    """
    _, bit_info = featurizer.morgan_fp_with_bit_info(mol)
    base = predict_blend_single(x_row, ep_idx, ep)
    scores: Dict[int, float] = defaultdict(float)
    on_bits = [b for b in bit_info if x_row[b] > 0.5]
    for b in tqdm(on_bits, desc="  ablating bits", unit="bit", leave=False):
        xp = x_row.copy()
        xp[b] = 0.0
        delta = base - predict_blend_single(xp, ep_idx, ep)   # >0 = removal hurt
        atoms = get_bit_atoms(mol, bit_info, b)
        if atoms:
            for a in atoms:
                scores[a] += delta / len(atoms)
    return scores


_ep_idx, _ep = 0, EPS[0]
if CONFIG.feature_set == "descriptors_only":
    print("feature_set='descriptors_only' -> no fingerprint bits to attribute; skipped.")
elif len(top_hits) == 0:
    print("No in-domain compounds to attribute.")
else:
    n_show = min(4, len(top_hits))
    fig, axes = plt.subplots(1, n_show, figsize=(4.2 * n_show, 4.4))
    axes = np.atleast_1d(axes)
    cmap = plt.get_cmap("coolwarm")
    for ax, ridx in zip(axes, tqdm(list(top_hits.index[:n_show]),
                                    desc="Atom-contribution maps", unit="mol")):
        mol = screen_feat["mols"][ridx]
        scores = compute_atom_sensitivities(mol, X_screen[ridx], _ep_idx, _ep)
        if not scores:
            ax.set_title("no 'on' bits"); ax.axis("off"); continue
        mx = max(abs(v) for v in scores.values()) or 1.0
        colors = {a: cmap(0.5 + 0.5 * (v / mx))[:3] for a, v in scores.items()}
        ax.imshow(Draw.MolToImage(mol, size=(320, 320), highlightAtoms=list(scores),
                                  highlightAtomColors=colors))
        ax.axis("off")
        cid = (screen_results.loc[ridx, "compound_id"]
               if "compound_id" in screen_results.columns else ridx)
        ax.set_title(f"{cid}\nP(active) {screen_results.loc[ridx, 'p_active_combined']:.3f}",
                     fontsize=9)
    plt.suptitle(f"Atom contributions to calibrated P(active) on {_ep} by substructure "
                 "ablation (red = raises, blue = lowers)", y=1.03)
    plt.tight_layout()
    plt.show()

## 17. Part F — Save outputs

Everything needed to reproduce, audit or redeploy: the shortlist and full library
predictions, the multi-task dataset with its source tags and per-endpoint label provenance,
all model families per endpoint with their bootstrap ensembles, the **probability
calibrators**, the shared feature scaler, the descriptor selector, the applicability-domain
object, and every evaluation table.

The **out-of-fold predictions are saved raw and calibrated**. Every pooled number in Part
E.2 derives from them, so the enrichment, precision@N and calibration figures can be
recomputed — or challenged — without re-running the folds.

F7 writes to its own directory. F6 left its artifacts beside F5's in a single folder, which
made it impossible to tell which run a file came from.

In [ ]:
# --- Save everything ----------------------------------------------------------------------
import joblib

# F7 writes to its own directory. F6 left its artifacts beside F5's in one folder, which
# made it impossible to tell which run a file came from.
out_dir = Path(CONFIG.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {out_dir}")

screen_results.to_csv(out_dir / "screening_predictions.csv", index=False)
top_hits.to_csv(out_dir / "shortlist.csv", index=False)
train_df.to_csv(out_dir / "psma_multitask_dataset.csv", index=False)

loco_eval_df.to_csv(out_dir / "loco_per_fold.csv", index=False)
loco_folds_df.to_csv(out_dir / "loco_fold_definitions.csv")
pooled_df.to_csv(out_dir / "loco_pooled.csv")
macro_df.to_csv(out_dir / "loco_macro.csv")
random_eval_df.to_csv(out_dir / "random_split_reference.csv", index=False)
ref_eval_df.to_csv(out_dir / "reference_split_scores.csv")
recalibration_df.to_csv(out_dir / "recalibration_before_after.csv")
if len(ablation_df):
    ablation_df.to_csv(out_dir / "transfer_learning_ablation.csv")
if len(derived_report):
    derived_report.to_csv(out_dir / "derived_endpoint_report.csv", index=False)

# out-of-fold predictions, raw and calibrated — every pooled number derives from these,
# so the enrichment, precision@N and calibration figures can be recomputed or challenged
# without re-running the folds
_oof = pd.DataFrame({"smiles": train_df["smiles"].values,
                     "source_tag": train_df["source_tag"].values,
                     "butina_cluster": butina_groups, "fold": fold_of})
for t, ep in enumerate(EPS):
    _oof[f"y_{ep}"] = Y_train_full[:, t]
for name in MODEL_ORDER:
    for ep in EPS:
        _oof[f"raw::{name}::{ep}"] = oof[name][ep]
        _oof[f"cal::{name}::{ep}"] = oof_cal[name][ep]
_oof.to_csv(out_dir / "out_of_fold_predictions.csv", index=False)

torch.save({
    "trunk_state_dict": {k: v.cpu() for k, v in nn_model_deploy.trunk.state_dict().items()},
    "task_heads_state_dict": {k: v.cpu()
                              for k, v in nn_model_deploy.task_heads.state_dict().items()},
    "hidden_dims": nn_hidden_dims_final,
    "dropout": nn_best_params["dropout"],
    "endpoints": EPS,
    "active_thresholds": {ep: CONFIG.active_thresholds[ep] for ep in EPS},
    "class_names": CLASS_NAMES,
    "used_pretraining": bool(CONFIG.use_pretraining),
}, out_dir / "nn_multitask_classifier.pt")

for ep in EPS:
    if gbm_deploy.get(ep) is not None:
        joblib.dump(gbm_deploy[ep], out_dir / f"lightgbm_{ep}.joblib")
        joblib.dump(gbm_boot.get(ep, []), out_dir / f"lightgbm_{ep}_bootstrap.joblib")
    if kpls_deploy.get(ep) is not None:
        joblib.dump(kpls_deploy[ep], out_dir / f"kpls_da_{ep}.joblib")
        joblib.dump(kpls_boot.get(ep, []), out_dir / f"kpls_da_{ep}_bootstrap.joblib")

joblib.dump(final_calibrators, out_dir / "probability_calibrators.joblib")

# A joblib dump of `_Calibrator` can only be LOADED in a session where that class is
# defined, which for a notebook-defined class means re-running this notebook. Export the
# maps as plain data as well, so the score -> probability mapping is reusable from
# anywhere: isotonic as its (x, y) knots for linear interpolation, Platt as coef/intercept.
_cal_export = {}
for (name, ep), cal in final_calibrators.items():
    entry = {"method": cal.method, "fitted": cal._m is not None}
    m = cal._m
    if m is not None:
        if cal.method == "isotonic":
            entry["x"] = [float(v) for v in np.asarray(m.X_thresholds_).ravel()]
            entry["y"] = [float(v) for v in np.asarray(m.y_thresholds_).ravel()]
            entry["apply"] = "numpy.interp(score, x, y), clipped to [0, 1]"
        else:
            entry["coef"] = float(np.ravel(m.coef_)[0])
            entry["intercept"] = float(np.ravel(m.intercept_)[0])
            entry["apply"] = "1 / (1 + exp(-(coef * score + intercept)))"
    _cal_export[f"{name}::{ep}"] = entry
with open(out_dir / "probability_calibrators.json", "w") as fh:
    json.dump(_cal_export, fh, indent=2)
joblib.dump(feature_scaler, out_dir / "feature_scaler.joblib")
joblib.dump(descriptor_selector, out_dir / "descriptor_selector.joblib")
joblib.dump(ad, out_dir / "applicability_domain.joblib")

with open(out_dir / "config.json", "w") as fh:
    json.dump({k: (list(v) if isinstance(v, (tuple, list)) else v)
               for k, v in CONFIG.__dict__.items()}, fh, indent=2, default=str)
with open(out_dir / "tuning_summary.json", "w") as fh:
    json.dump({"gbm_best_params": gbm_best_params,
               "kpls_best_params": kpls_best_params,
               "nn_best_params": nn_best_params,
               "feature_set": CONFIG.feature_set,
               "use_pretraining": CONFIG.use_pretraining,
               "recalibration": CONFIG.recalibration}, fh, indent=2, default=str)

print(f"\nSaved {len(list(out_dir.iterdir()))} files:")
for p in sorted(out_dir.iterdir()):
    print(f" - {p.name}")

## 18. Notes for reuse

### Reading the results

- **Quote the pooled leave-one-cluster-out numbers.** Not the random-split reference (kept
  only to make the leakage gap visible), and not the macro average (kept only to show the
  spread across families). Pooled scores every compound once on a prediction made while its
  whole family was absent from training.
- **For a screen, read `precision@N` and `EF` before `roc_auc`.** AUC can look respectable
  while the top of the ranking is useless, and the top of the ranking is the only part
  anyone acts on.
- **Ignore `EF@1%` on small folds.** At n = 30 the top 1% is one compound. The pooled row
  is the only place the enrichment figures carry information.
- **A fold that fails is a finding.** The per-fold table names the family the model could
  not reach. Averaging that away is what made the F4 numbers misleading.
- **Check the AD count before reading the shortlist at all.** If almost nothing in the
  library is in domain, the ranking is a list of least-extrapolated guesses and the real
  conclusion is about the library.

### Knobs

| setting | default | effect |
|---|---|---|
| `active_thresholds` | 8.0 / 8.0 | 10 nM on both endpoints. Change *before* looking at results, then re-run Part A.2b to confirm the classes stay usable. |
| `drop_derived_endpoint_values` | True | Removes pKi labels that are arithmetic transforms of pIC50. Turning it off restores F3–F5 behaviour and the endpoint-disagreement problem with it. |
| `use_pretraining` | False | See Part E.2d. `run_pretraining_ablation = True` re-measures it. |
| `recalibration` | "sigmoid" | Platt scaling: 2 parameters, strictly monotone, cannot collapse ties. Try `"isotonic"` only if the before/after table shows a shape a sigmoid cannot follow — it overfits small folds into step functions. |
| `ad_tanimoto_veto` | True | Makes structural dissimilarity decisive. Off reproduces F5's much larger and much less trustworthy in-domain set. |
| `loco_min_cluster_size` / `loco_min_fold_actives` | 25 / 5 | Raise to get fewer, more reliable folds; lower to see more families individually at the cost of unscoreable rows. |
| `downweight_dominant_source` | False | Inverse-sqrt-frequency weights per filing. Compare the pooled numbers both ways before adopting. |
| `butina_cutoff` | 0.4 | Higher makes families coarser and the held-out test harder. |

### A new target instead of PSMA

Change `target_uniprot_id` / `target_name_like` / `organism_like`, and
`chembl_pretrain_target_keywords` for that target's related family. Then **run Part A.2b
and A.2c before anything else**: the thresholds, the class balance, and whether the two
endpoints are independent measurements are all target-specific, and all three were
wrong-by-default for PSMA.

### Still outstanding

- **The screening library is the wrong file.** Its first entries are butyl chloride,
  β-propiolactone and diethylene glycol. In F6 only 36 of 289,554 compounds were within
  structural reach of the training set. This is the binding constraint on the entire
  exercise, and no change inside this notebook can work around it. A drug-like commercial
  set, or better a glutamate–urea-focused sub-library, is worth more than every modelling
  improvement in F3–F7 combined.
- **Nothing has been validated in a lab.** Every number here is the model grading its own
  homework under progressively stricter exam conditions. The pooled figures should be read
  as a prediction of the hit rate, and the only thing that settles it is measuring a
  handful of compounds.
- **712 compounds is still small and still narrow.** One filing supplies roughly a sixth of
  them, and the largest Butina cluster is close to a fifth. PSMA actives from ChEMBL that
  are absent from BindingDB, and property-matched decoys as presumed inactives, would both
  add more than any further modelling change.
- **The AutoQSAR gap penalty prefers under-fit models.** It is retained for continuity with
  the Dixon et al. method, and the tuning cell now says explicitly when the model it chose
  was not the one with the better held-out score. Consider selecting on held-out AUC
  directly.